# Agenti AI con Python: dal tool calling a LangGraph

Questo notebook riunisce in **un unico file** la teoria e la pratica dei laboratori sugli agenti AI. Segue lo stesso schema di `lezione_oggi`: prima si spiega il concetto, poi si scrive il codice, poi lo si prova.

Il filo conduttore è un unico **caso guida**: l'**Assistente Acquisti Hardware** di un'azienda.

> Un dipendente invia una richiesta di acquisto in linguaggio naturale (es. *"Sono Mario Rossi, badge E4521, posso comprare il ThinkPad?"*).
> L'assistente controlla budget, catalogo e policy, poi decide: **approvazione automatica**, **escalation al manager** oppure **rifiuto motivato**.

Lo stesso problema viene risolto con strumenti sempre più evoluti, così si vede **perché** esiste ogni pezzo.

## Come è organizzato

| Parte | Cosa impari | Notebook di origine |
| :--- | :--- | :--- |
| **0** | Setup, dati di esempio, regole di business | — |
| **1** | Cos'è un agente, tool calling, loop ReAct scritto a mano | — |
| **2** | Protocolli **MCP** (agente ➜ tool) e **A2A** (agente ➜ agente) | `01_MCP_Tools_Hardware_Assistant`, `02_A2A_Protocol_MultiAgent` |
| **3** | **LangChain**: `@tool`, prompt, `AgentExecutor`, Human Approval, Agent-as-Tool, MVP | `01_Architettura_Agente_Langflow`, `02_Agente_con_Tools_Ollama`, `03_Pattern_Human_Approval`, `04_Agente_as_Tool`, `05_Esercitazione_MVP` |
| **4** | **LangGraph**: stato, nodi, archi, cicli, ReAct, Supervisor, persistenza, Human-in-the-Loop | `01_Pattern_Orchestrazione`, `02_LangGraph_MVP`, `03_LangGraph_HumanInLoop` |
| **5** | Esercitazione finale: orchestrazione completa con retry, failure path e approvazione umana | `04_Esercitazione_Orchestrator` |
| **6** | Riepilogo, tabelle di confronto, errori comuni | — |

#### Legenda delle celle

- 🧠 **richiede Ollama**: la cella chiama un LLM locale (`llama3.2`). Se Ollama non è raggiungibile viene saltata con un avviso, senza bloccare "Esegui tutto".
- ⚙️ **gira senza LLM**: logica deterministica, funziona sempre.

>NB) Le celle ⚙️ sono volutamente tante: **la parte deterministica di un agente va testata senza LLM**. Il modello serve solo dove serve davvero il linguaggio naturale.

# Parte 0 — Setup e dati di esempio

## Dipendenze

Servono Python 3.10+ e questi pacchetti:

| Pacchetto | A cosa serve |
| :--- | :--- |
| `langchain-core` | tool, prompt, messaggi, callback |
| `langchain-ollama` | classe `ChatOllama` (modello locale) |
| `langchain-classic` | `AgentExecutor` e `create_tool_calling_agent` (in LangChain ≥ 1.0 sono stati spostati qui) |
| `langgraph` | grafi con stato, checkpoint, interrupt |
| `requests` | chiamate HTTP dirette a Ollama (Parte 1) |

La cella seguente installa solo quello che manca.

In [ ]:
# Verifica e installa le dipendenze mancanti
import importlib.util
import subprocess
import sys

# nome del modulo da importare -> nome del pacchetto da installare con pip
dipendenze = {
    "requests": "requests",
    "langchain_core": "langchain-core",
    "langchain_ollama": "langchain-ollama",
    "langchain_classic": "langchain-classic",
    "langgraph": "langgraph",
}

mancanti = [pkg for modulo, pkg in dipendenze.items() if importlib.util.find_spec(modulo) is None]

if mancanti:
    print("Installazione in corso:", mancanti)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + mancanti)
    print("Fatto. Se compaiono errori di import, riavvia il kernel e riesegui.")
else:
    print("Tutte le dipendenze sono già installate.")

## Connessione a Ollama

**Ollama** esegue modelli open source in locale: nessuna API key, nessun costo. Prima di usare le celle 🧠 servono due comandi nel terminale:

```bash
ollama serve          # avvia il server (porta 11434)
ollama pull llama3.2  # scarica il modello, una sola volta
```

La cella seguente controlla se il server risponde e imposta la variabile `OLLAMA_OK`, che useremo come "interruttore" per le celle con LLM.

In [ ]:
import requests

OLLAMA_URL = "http://localhost:11434"
MODEL_NAME = "llama3.2"
OLLAMA_OK = False   # diventa True se il server risponde e il modello è presente

AVVISO_OLLAMA = (
    "⚠️  Ollama non raggiungibile (o modello assente): cella 🧠 saltata.\n"
    f"    Avvia 'ollama serve', esegui 'ollama pull {MODEL_NAME}' e riesegui la cella di verifica."
)

try:
    risposta = requests.get(f"{OLLAMA_URL}/api/tags", timeout=3)
    modelli = [m["name"] for m in risposta.json().get("models", [])]
    OLLAMA_OK = any(MODEL_NAME in nome for nome in modelli)
    print("Ollama raggiungibile. Modelli:", modelli)
    print(f"✅ {MODEL_NAME} presente" if OLLAMA_OK else f"⚠️  {MODEL_NAME} non trovato -> ollama pull {MODEL_NAME}")
except Exception as errore:
    print(AVVISO_OLLAMA)
    print("   Dettaglio:", errore)

## Dati di esempio

Un agente ragiona su **dati**. Qui li teniamo fuori dal codice, in una cartella `data/`, come farebbe un'applicazione vera:

| File | Contenuto |
| :--- | :--- |
| `employees.json` | dipendenti: ruolo, dipartimento, budget residuo, progetto attivo, email del manager |
| `products.json` | catalogo: nome, categoria, prezzo, ruoli target |
| `policy_hardware.md` | regole aziendali in linguaggio naturale |

>NB) Questi dati sono **di esempio**, ricostruiti dagli scenari dei laboratori (Mario Rossi, Giulia Verdi, ThinkPad P16s, monitor Dell UltraSharp...). La cella li crea da zero, quindi il notebook è autosufficiente.

In [ ]:
import json
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

employees = {
    "E4521": {"name": "Mario Rossi", "department": "Engineering", "role": "Senior Developer",
              "budget_residuo": 1800, "active_project": "Project Delta",
              "manager_email": "anna.bianchi@azienda.it"},
    "E1082": {"name": "Giulia Verdi", "department": "Marketing", "role": "Marketing Specialist",
              "budget_residuo": 450, "active_project": "Brand Campaign Q2",
              "manager_email": "paolo.neri@azienda.it"},
    "E9900": {"name": "Alessandro Corti", "department": "Sales", "role": "Account Manager",
              "budget_residuo": 600, "active_project": None,
              "manager_email": "laura.gialli@azienda.it"},
}

products = {
    "LAP-DEV-01": {"name": "ThinkPad P16s Gen 2", "category": "laptop", "price_eur": 1650,
                   "target_roles": ["Engineering", "Data Science"], "availability": "disponibile"},
    "LAP-STD-01": {"name": "Dell Latitude 5450", "category": "laptop", "price_eur": 950,
                   "target_roles": ["Marketing", "Sales", "HR", "Amministrazione"], "availability": "disponibile"},
    "MON-4K-01": {"name": "Dell UltraSharp 27 4K", "category": "monitor", "price_eur": 520,
                  "target_roles": ["Engineering", "Marketing"], "availability": "disponibile"},
    "ACC-DOCK-01": {"name": "Thunderbolt 4 Dock", "category": "accessory", "price_eur": 190,
                    "target_roles": ["Engineering", "Marketing", "Sales"], "availability": "disponibile"},
}

policy_md = """# Policy Acquisti Hardware

## Laptop
- Fascia Engineering / Tech / Data Science: massimo 1800 EUR.
- Fascia Marketing / HR / Sales / Amministrazione: massimo 1000 EUR.
- Il prezzo deve essere coperto dal budget residuo del dipendente.

## Monitor
- Fino a 300 EUR: approvazione automatica se coperto dal budget residuo.
- Oltre 300 EUR: serve l'approvazione del manager (escalation).

## Accessory (accessori)
- Fino a 300 EUR: approvazione automatica se coperto dal budget residuo.
- Oltre 300 EUR: serve l'approvazione del manager (escalation).

## Regole generali
- Dipendente non trovato: RIFIUTO.
- Prodotto non presente a catalogo: RIFIUTO.
- Prezzo superiore al budget residuo: ESCALATION al manager.
"""

(DATA_DIR / "employees.json").write_text(json.dumps(employees, indent=2, ensure_ascii=False), encoding="utf-8")
(DATA_DIR / "products.json").write_text(json.dumps(products, indent=2, ensure_ascii=False), encoding="utf-8")
(DATA_DIR / "policy_hardware.md").write_text(policy_md, encoding="utf-8")

# Carichiamo i dati in memoria: li useremo in tutto il notebook
EMPLOYEES_DB = json.loads((DATA_DIR / "employees.json").read_text(encoding="utf-8"))
PRODUCTS_DB = json.loads((DATA_DIR / "products.json").read_text(encoding="utf-8"))
POLICY_TEXT = (DATA_DIR / "policy_hardware.md").read_text(encoding="utf-8")

print(f"Dipendenti: {list(EMPLOYEES_DB)}")
print(f"Prodotti:   {list(PRODUCTS_DB)}")
print(f"Policy:     {len(POLICY_TEXT)} caratteri")

## Le regole di business vivono nel codice, non nel modello

Un errore tipico dei principianti è chiedere al LLM di fare confronti numerici ("1650 è minore di 1800?"). I modelli piccoli sbagliano, e un errore su un acquisto è un problema vero.

| Compito | Chi lo fa | Perché |
| :--- | :--- | :--- |
| Capire cosa chiede l'utente, scrivere la risposta | **LLM** | è bravo col linguaggio |
| Confronti numerici, soglie, regole di policy | **Codice Python** | deterministico e testabile |
| Decidere *quale* tool chiamare e con quali argomenti | **LLM** | è il suo ruolo di "ragionatore" |
| Eseguire il tool | **Codice Python** | il modello non esegue nulla |

Per questo definiamo due funzioni di supporto, che useremo in tutte le parti successive:

- `cerca_prodotti(query)`: ricerca nel catalogo per parole intere (nome o categoria), restituendo solo i migliori risultati;
- `applica_policy(emp, prod)`: restituisce `("APPROVATO" | "ESCALATION", motivazione)`.

In [ ]:
import re

# ── Regole di business (parametri) ───────────────────────────────────────
FASCE_LAPTOP = {"Engineering": 1800, "Tech": 1800, "Data Science": 1800}
FASCIA_LAPTOP_DEFAULT = 1000      # Marketing, HR, Sales, Amministrazione
SOGLIA_MONITOR_ACCESSORI = 300

STOPWORDS = {
    "il", "lo", "la", "i", "gli", "le", "un", "una", "di", "del", "della", "da", "per", "con", "e",
    "ciao", "sono", "vorrei", "voglio", "posso", "acquistare", "comprare", "ho", "bisogno", "mi", "serve",
    "salve", "the", "for", "and", "or",
}


def tokenizza(testo: str) -> list[str]:
    """Spezza un testo in parole minuscole, senza punteggiatura né stopword."""
    parole = re.findall(r"[a-z0-9àèéìòù]+", testo.lower())
    return [p for p in parole if len(p) >= 2 and p not in STOPWORDS]


def cerca_prodotti(query: str) -> dict:
    """Cerca nel catalogo per PAROLE INTERE (nome o categoria).
    Restituisce solo i prodotti con il punteggio più alto (dizionario vuoto se nessuno)."""
    token = tokenizza(query)
    punteggi = {}
    for pid, prod in PRODUCTS_DB.items():
        parole_prodotto = set(tokenizza(prod["name"])) | {prod["category"]}
        punteggio = sum(1 for t in token if t in parole_prodotto)
        if punteggio > 0:
            punteggi[pid] = punteggio
    if not punteggi:
        return {}
    migliore = max(punteggi.values())
    return {pid: PRODUCTS_DB[pid] for pid, p in punteggi.items() if p == migliore}


def applica_policy(emp: dict, prod: dict) -> tuple[str, str]:
    """Applica la policy aziendale. Restituisce (esito, motivazione)."""
    prezzo, budget, categoria = prod["price_eur"], emp["budget_residuo"], prod["category"]

    if categoria in ("monitor", "accessory") and prezzo > SOGLIA_MONITOR_ACCESSORI:
        return "ESCALATION", f"{categoria} da {prezzo}€ supera la soglia di {SOGLIA_MONITOR_ACCESSORI}€"

    if categoria == "laptop":
        fascia = FASCE_LAPTOP.get(emp["department"], FASCIA_LAPTOP_DEFAULT)
        if prezzo > fascia:
            return "ESCALATION", f"laptop da {prezzo}€ supera la fascia di {fascia}€ per {emp['department']}"

    if prezzo > budget:
        return "ESCALATION", f"prezzo {prezzo}€ superiore al budget residuo di {budget}€"

    return "APPROVATO", f"prezzo {prezzo}€ entro fascia e budget residuo di {budget}€"


print("Funzioni di supporto definite.")

### ⚙️ Testiamo le regole (senza LLM)

Prima di costruire qualsiasi agente verifichiamo che le regole diano gli esiti attesi dei tre scenari classici. Le `assert` si fermano con un errore se qualcosa non torna.

In [ ]:
# Scenario 1: Mario Rossi (Engineering, budget 1800) vuole il ThinkPad da 1650€ -> APPROVATO
esito, motivo = applica_policy(EMPLOYEES_DB["E4521"], PRODUCTS_DB["LAP-DEV-01"])
print("Mario  :", esito, "-", motivo)
assert esito == "APPROVATO"

# Scenario 2: Giulia Verdi (Marketing, budget 450) vuole il monitor da 520€ -> ESCALATION
esito, motivo = applica_policy(EMPLOYEES_DB["E1082"], PRODUCTS_DB["MON-4K-01"])
print("Giulia :", esito, "-", motivo)
assert esito == "ESCALATION"

# Scenario 3: prodotto inesistente -> la ricerca non trova nulla
print("Vision Pro ->", cerca_prodotti("Apple Vision Pro"))
assert cerca_prodotti("Apple Vision Pro") == {}

# Ricerca per parole intere: 'thinkpad laptop' deve dare SOLO il ThinkPad (punteggio 2)
print("thinkpad laptop ->", list(cerca_prodotti("thinkpad laptop")))
assert list(cerca_prodotti("thinkpad laptop")) == ["LAP-DEV-01"]

print("\nTutte le regole funzionano.")

# Parte 1 — Fondamenti: LLM, agenti e tool calling

## Cos'è un agente?

Un **LLM** da solo prende testo e restituisce testo: non può leggere un database, né inviare una mail. Un **agente** è un LLM inserito in un programma che gli dà **strumenti** (*tool*) e lo fa ragionare in un ciclo, finché il compito non è concluso.

| | Chatbot | Agente |
| :--- | :--- | :--- |
| Input → output | una domanda, una risposta | un obiettivo, **più passi** |
| Dati esterni | solo quelli nel prompt | li recupera da sé con i tool |
| Azioni | nessuna | può eseguire funzioni (cercare, scrivere, creare ticket) |
| Controllo del flusso | nessuno | il modello **decide** il prossimo passo |

I componenti di un agente sono quattro:

- **LLM**: il ragionatore;
- **Prompt di sistema**: ruolo, regole e policy;
- **Tool**: funzioni che il modello può richiedere;
- **Stato / memoria**: la conversazione e i risultati intermedi.

## Il ciclo agentico (ReAct)

Il modello **non esegue** i tool: li *richiede*. Chi li esegue è il nostro codice (l'*host*).

```
        ┌───────────────────────────────────────────────┐
        │                                               │
  utente ──► [ LLM ] ──► vuole un tool? ──sì──► host esegue il tool
                │              │                         │
                │              no                        │
                ▼              ▼                         │
          risposta finale   (fine)        risultato ◄────┘
                                              │
                                              └──► torna all'LLM
```

Questo schema si chiama **ReAct** (*Reason + Act*): ragiona, agisci, osserva il risultato, ripeti. Si ferma quando il modello risponde senza richiedere altri tool.

>NB) Un limite di iterazioni (`max_iterations`) è **obbligatorio** in ogni loop agentico: senza, un modello confuso può chiamare tool all'infinito.

## Anatomia di una tool call

Per far "vedere" un tool al modello gli passiamo uno **schema JSON** con nome, descrizione e parametri. La **descrizione** è fondamentale: è l'unica cosa che il modello legge per capire *quando* usare il tool.

```json
{
  "type": "function",
  "function": {
    "name": "get_employee_budget",
    "description": "Restituisce budget residuo e ruolo di un dipendente dato il badge ID.",
    "parameters": {
      "type": "object",
      "properties": {"employee_id": {"type": "string", "description": "Badge ID, es. 'E4521'"}},
      "required": ["employee_id"]
    }
  }
}
```

Quando il modello decide di usarlo, **non scrive testo**: risponde con una richiesta strutturata.

```json
{"role": "assistant", "content": "",
 "tool_calls": [{"function": {"name": "get_employee_budget", "arguments": {"employee_id": "E4521"}}}]}
```

L'host esegue la funzione e rimanda il risultato al modello con ruolo `tool`:

```json
{"role": "tool", "tool_name": "get_employee_budget", "content": "{\"status\": \"OK\", \"budget_residuo\": 1800}"}
```

In [ ]:
# ── Due tool scritti come semplici funzioni Python ───────────────────────

def get_employee_budget(employee_id: str) -> dict:
    """Restituisce budget residuo, ruolo, dipartimento e manager di un dipendente."""
    emp = EMPLOYEES_DB.get(employee_id)
    if emp is None:
        return {"status": "NOT_FOUND", "message": f"Dipendente '{employee_id}' non trovato."}
    return {"status": "OK", "employee_id": employee_id, **emp}


def search_product(query: str) -> dict:
    """Cerca prodotti nel catalogo per nome o categoria."""
    trovati = cerca_prodotti(query)
    if not trovati:
        return {"status": "NOT_FOUND", "message": f"Nessun prodotto per '{query}': NON è a catalogo."}
    return {"status": "OK", "risultati": trovati}


# ── Lo schema JSON, scritto A MANO ───────────────────────────────────────
TOOLS_SCHEMA_MANUALE = [
    {"type": "function", "function": {
        "name": "get_employee_budget",
        "description": "Restituisce budget residuo, ruolo e dipartimento di un dipendente dato il suo badge ID. "
                       "Chiamare SEMPRE prima di valutare un acquisto.",
        "parameters": {"type": "object",
                       "properties": {"employee_id": {"type": "string", "description": "Badge ID, es. 'E4521'"}},
                       "required": ["employee_id"]}}},
    {"type": "function", "function": {
        "name": "search_product",
        "description": "Cerca un prodotto nel catalogo hardware aziendale e ne restituisce il prezzo ufficiale.",
        "parameters": {"type": "object",
                       "properties": {"query": {"type": "string", "description": "Nome o categoria, es. 'ThinkPad', 'monitor'"}},
                       "required": ["query"]}}},
]

# ── Registro: nome del tool -> funzione Python ───────────────────────────
REGISTRO_TOOL = {"get_employee_budget": get_employee_budget, "search_product": search_product}


def esegui_tool_locale(nome: str, argomenti: dict) -> dict:
    """Esegue il tool richiesto dal modello. Gestisce i nomi inventati (allucinazioni)."""
    if nome not in REGISTRO_TOOL:
        return {"status": "ERROR", "message": f"Il tool '{nome}' non esiste. Disponibili: {list(REGISTRO_TOOL)}"}
    return REGISTRO_TOOL[nome](**argomenti)


# Proviamo i tool direttamente, senza LLM
print(esegui_tool_locale("get_employee_budget", {"employee_id": "E4521"}))
print(esegui_tool_locale("tool_inventato", {}))

### 🧠 Prima chiamata: il modello richiede un tool

Inviamo una domanda a Ollama insieme agli schemi dei tool. Osserva che la risposta **non contiene testo**, ma una `tool_calls`.

In [ ]:
if OLLAMA_OK:
    payload = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": "Qual è il budget residuo del dipendente E4521?"}],
        "tools": TOOLS_SCHEMA_MANUALE,
        "stream": False,
        "options": {"temperature": 0},   # 0 = risposte il più possibile deterministiche
    }
    risposta = requests.post(f"{OLLAMA_URL}/api/chat", json=payload, timeout=120)
    risposta.raise_for_status()
    messaggio = risposta.json()["message"]
    print("Testo:      ", repr(messaggio.get("content")))
    print("Tool calls: ", json.dumps(messaggio.get("tool_calls"), indent=2, ensure_ascii=False))
else:
    print(AVVISO_OLLAMA)

## Scriviamo noi il loop

Ora mettiamo insieme il ciclo ReAct: chiamiamo il modello, se chiede tool li eseguiamo, rimandiamo i risultati, e ripetiamo fino alla risposta finale.

La funzione è **generica**: riceve gli schemi dei tool e una funzione `esegui_tool`. Nella Parte 2 le passeremo tool che arrivano da un server MCP, senza cambiare una riga del loop.

In [ ]:
def agent_loop(domanda: str, system_prompt: str, tools_schema: list, esegui_tool,
               max_iterazioni: int = 6, verbose: bool = True) -> str:
    """Loop agentico scritto a mano: LLM -> (tool -> LLM)* -> risposta finale."""
    messaggi = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": domanda},
    ]
    if verbose:
        print(f"👤 UTENTE: {domanda}\n")

    for passo in range(1, max_iterazioni + 1):
        payload = {"model": MODEL_NAME, "messages": messaggi, "tools": tools_schema,
                   "stream": False, "options": {"temperature": 0}}
        risposta = requests.post(f"{OLLAMA_URL}/api/chat", json=payload, timeout=120)
        risposta.raise_for_status()
        msg = risposta.json()["message"]
        messaggi.append(msg)                      # la risposta del modello entra nella cronologia

        richieste = msg.get("tool_calls") or []
        if not richieste:                         # nessun tool richiesto = risposta finale
            if verbose:
                print(f"🤖 RISPOSTA FINALE:\n{msg.get('content', '')}")
            return msg.get("content", "")

        for richiesta in richieste:
            nome = richiesta["function"]["name"]
            argomenti = richiesta["function"]["arguments"]
            risultato = esegui_tool(nome, argomenti)
            if verbose:
                print(f"[passo {passo}] 🔧 {nome}({argomenti}) -> {json.dumps(risultato, ensure_ascii=False)[:150]}")
            messaggi.append({
                "role": "tool",
                "tool_name": nome,        # campo usato dalle versioni recenti di Ollama
                "name": nome,             # compatibilità con versioni precedenti
                "content": json.dumps(risultato, ensure_ascii=False),
            })

    print("⚠️  Raggiunto il limite di iterazioni.")
    return ""


SYSTEM_PROMPT_BASE = f"""Sei l'Assistente Acquisti Hardware aziendale.

Procedura obbligatoria:
1. Chiama 'get_employee_budget' con il badge ID del dipendente.
2. Chiama 'search_product' per conoscere il prezzo ufficiale del prodotto. Se non è a catalogo, RIFIUTA.
3. Confronta prezzo, budget residuo e policy. Rispondi in italiano con APPROVATO, ESCALATION AL MANAGER o RIFIUTO e la motivazione.

=== POLICY ===
{POLICY_TEXT}
=============
"""
print("Loop definito.")

In [ ]:
if OLLAMA_OK:
    print("═" * 60, "\nCASO 1 — Mario Rossi, atteso: APPROVATO\n" + "═" * 60)
    agent_loop("Ciao, sono Mario Rossi (badge E4521). Posso acquistare il ThinkPad P16s?",
               SYSTEM_PROMPT_BASE, TOOLS_SCHEMA_MANUALE, esegui_tool_locale)

    print("\n" + "═" * 60, "\nCASO 2 — Giulia Verdi, atteso: ESCALATION\n" + "═" * 60)
    agent_loop("Sono Giulia Verdi (badge E1082). Posso comprare il monitor 4K Dell UltraSharp?",
               SYSTEM_PROMPT_BASE, TOOLS_SCHEMA_MANUALE, esegui_tool_locale)
else:
    print(AVVISO_OLLAMA)

## Cosa abbiamo imparato (e cosa è scomodo)

- Il modello **richiede** i tool, il nostro codice li **esegue**: la sicurezza sta in chi esegue.
- Il loop ReAct sono poche righe: messaggi ➜ modello ➜ eventuali tool ➜ di nuovo modello.
- Però abbiamo scritto **a mano** gli schemi JSON dei tool, e il loop è legato a un solo modello e a un solo modo di esporre i tool.

Questi due problemi hanno due soluzioni, che vediamo nelle parti successive:

| Problema | Soluzione | Parte |
| :--- | :--- | :--- |
| Ogni app reinventa il modo di collegare tool e modelli | **Protocolli standard** (MCP, A2A) | 2 |
| Schemi a mano, loop da scrivere, nessuna osservabilità | **Framework** (LangChain, LangGraph) | 3 e 4 |

# Parte 2 — Protocolli: MCP e A2A

## Il problema N×M

Immagina **N** applicazioni AI (chat, IDE, agenti interni...) e **M** sistemi da collegare (database, CRM, calendario...). Senza uno standard, ogni coppia richiede un'integrazione su misura: **N × M** integrazioni.

Con un protocollo comune ogni applicazione parla il protocollo una volta sola, e ogni sistema lo espone una volta sola: **N + M** integrazioni.

```
   SENZA STANDARD                          CON UNO STANDARD
 App1 ──┬── DB                          App1 ──┐
 App1 ──┼── CRM                         App2 ──┼──► [ PROTOCOLLO ] ◄──┬── DB
 App2 ──┼── DB          (N × M)         App3 ──┘                      ├── CRM
 App2 ──┴── CRM                                        (N + M)       └── Calendario
```

#### Protocollo, framework, architettura: tre cose diverse

| Livello | Cos'è | Esempi |
| :--- | :--- | :--- |
| **Protocollo** | regole standard di comunicazione tra componenti | MCP, A2A, HTTP |
| **Framework** | libreria che aiuta a costruire il sistema | LangChain, LangGraph |
| **Architettura** | come combini i pezzi nel **tuo** sistema | agente singolo, agent-as-tool, supervisor |

## MCP — Model Context Protocol

MCP standardizza il collegamento tra **un agente e i suoi strumenti/dati**. Ha tre ruoli:

| Ruolo | Cos'è | Esempio |
| :--- | :--- | :--- |
| **Host** | l'applicazione che contiene il LLM | la nostra app, un IDE, una chat |
| **Client** | il connettore dentro l'host, parla col server | una libreria client MCP |
| **Server** | il programma che espone capacità | il server dell'ufficio acquisti |

Un server può esporre **tre primitive**:

| Primitiva | A cosa serve | Chi la controlla | Esempio nel caso guida |
| :--- | :--- | :--- | :--- |
| **Tools** | funzioni eseguibili | il **modello** decide quando chiamarle | `get_employee_budget`, `search_product` |
| **Resources** | dati in sola lettura da caricare nel contesto | l'**applicazione** | `policy://acquisti-hardware` |
| **Prompts** | template di istruzioni riutilizzabili | l'**utente** | "Valutazione standard di un acquisto" |

La comunicazione usa messaggi **JSON-RPC 2.0**, ad esempio `tools/list`, `tools/call`, `resources/read`. Il trasporto può essere `stdio` (processo locale) o HTTP.

>NB) Qui costruiamo un **mini server MCP didattico** direttamente nel notebook, con gli stessi nomi di metodi e la stessa forma dei messaggi del protocollo reale. Serve a vedere cosa succede "sotto il cofano". Per un'app vera si usa l'SDK ufficiale (vedi più sotto).

In [ ]:
import inspect
from typing import get_type_hints

TIPI_JSON = {str: "string", int: "integer", float: "number", bool: "boolean", dict: "object", list: "array"}


class ErroreMCP(Exception):
    def __init__(self, codice: int, messaggio: str):
        super().__init__(messaggio)
        self.codice, self.messaggio = codice, messaggio


class MiniMCPServer:
    """Server MCP didattico: registra Tools, Resources, Prompts e risponde a richieste JSON-RPC."""

    def __init__(self, nome: str):
        self.nome = nome
        self._tools, self._resources, self._prompts = {}, {}, {}

    # ── Registrazione tramite decoratori ─────────────────────────────────
    def tool(self):
        def decoratore(func):
            self._tools[func.__name__] = func
            return func
        return decoratore

    def resource(self, uri: str, descrizione: str = ""):
        def decoratore(func):
            self._resources[uri] = {"func": func, "descrizione": descrizione}
            return func
        return decoratore

    def prompt(self):
        def decoratore(func):
            self._prompts[func.__name__] = func
            return func
        return decoratore

    # ── Lo schema JSON viene GENERATO da type hint e docstring ───────────
    @staticmethod
    def _input_schema(func) -> dict:
        hints = get_type_hints(func)
        proprieta, obbligatori = {}, []
        for nome, parametro in inspect.signature(func).parameters.items():
            proprieta[nome] = {"type": TIPI_JSON.get(hints.get(nome, str), "string")}
            if parametro.default is inspect.Parameter.empty:
                obbligatori.append(nome)
        return {"type": "object", "properties": proprieta, "required": obbligatori}

    # ── Punto di ingresso: una richiesta JSON-RPC in, una risposta out ───
    def handle(self, richiesta: dict) -> dict:
        try:
            risultato = self._dispatch(richiesta.get("method"), richiesta.get("params", {}))
            return {"jsonrpc": "2.0", "id": richiesta.get("id"), "result": risultato}
        except ErroreMCP as e:
            return {"jsonrpc": "2.0", "id": richiesta.get("id"), "error": {"code": e.codice, "message": e.messaggio}}

    def _dispatch(self, metodo: str, params: dict) -> dict:
        if metodo == "tools/list":
            return {"tools": [{"name": n, "description": inspect.getdoc(f) or "",
                               "inputSchema": self._input_schema(f)} for n, f in self._tools.items()]}

        if metodo == "tools/call":
            nome = params.get("name")
            if nome not in self._tools:
                raise ErroreMCP(-32602, f"Tool sconosciuto: {nome}")
            try:
                esito = self._tools[nome](**params.get("arguments", {}))
                return {"content": [{"type": "text", "text": json.dumps(esito, ensure_ascii=False)}], "isError": False}
            except Exception as errore:      # un errore di esecuzione NON è un errore di protocollo
                return {"content": [{"type": "text", "text": f"Errore: {errore}"}], "isError": True}

        if metodo == "resources/list":
            return {"resources": [{"uri": u, "description": r["descrizione"], "mimeType": "text/markdown"}
                                  for u, r in self._resources.items()]}

        if metodo == "resources/read":
            uri = params.get("uri")
            if uri not in self._resources:
                raise ErroreMCP(-32602, f"Risorsa sconosciuta: {uri}")
            return {"contents": [{"uri": uri, "mimeType": "text/markdown", "text": self._resources[uri]["func"]()}]}

        if metodo == "prompts/list":
            return {"prompts": [{"name": n, "description": inspect.getdoc(f) or "",
                                 "arguments": [{"name": a} for a in inspect.signature(f).parameters]}
                                for n, f in self._prompts.items()]}

        if metodo == "prompts/get":
            nome = params.get("name")
            if nome not in self._prompts:
                raise ErroreMCP(-32602, f"Prompt sconosciuto: {nome}")
            testo = self._prompts[nome](**params.get("arguments", {}))
            return {"messages": [{"role": "user", "content": {"type": "text", "text": testo}}]}

        raise ErroreMCP(-32601, f"Metodo non trovato: {metodo}")


print("MiniMCPServer definito.")

### Il server del caso guida

Registriamo tre tool, una resource e un prompt. Nota che **non scriviamo nessuno schema JSON**: lo genera il server leggendo type hint e docstring. È lo stesso principio di `@mcp.tool()` nell'SDK reale.

In [ ]:
import datetime

server_mcp = MiniMCPServer("hardware-assistant")
TICKET_MCP = []   # "database" in memoria dei ticket creati


@server_mcp.tool()
def get_employee_budget(employee_id: str) -> dict:
    """Restituisce budget residuo, ruolo, dipartimento e manager di un dipendente dato il badge ID (es. 'E4521')."""
    emp = EMPLOYEES_DB.get(employee_id)
    if emp is None:
        return {"status": "NOT_FOUND", "message": f"Dipendente '{employee_id}' non trovato."}
    return {"status": "OK", "employee_id": employee_id, **emp}


@server_mcp.tool()
def search_product(query: str) -> dict:
    """Cerca prodotti nel catalogo hardware per nome o categoria (es. 'laptop', 'ThinkPad') e ne restituisce sku e prezzo."""
    trovati = cerca_prodotti(query)
    if not trovati:
        return {"status": "NOT_FOUND", "message": f"Nessun prodotto per '{query}': non è a catalogo."}
    return {"status": "OK", "risultati": {sku: {"name": p["name"], "category": p["category"], "price_eur": p["price_eur"]}
                                          for sku, p in trovati.items()}}


@server_mcp.tool()
def request_manager_approval(employee_id: str, sku: str, motivazione: str) -> dict:
    """Crea un ticket di approvazione per il manager. Da usare SOLO se l'acquisto supera budget o policy."""
    ticket = {"ticket_id": f"TKT-{len(TICKET_MCP) + 1:04d}", "employee_id": employee_id, "sku": sku,
              "motivazione": motivazione, "status": "PENDING", "created_at": datetime.datetime.now().isoformat()}
    TICKET_MCP.append(ticket)
    return ticket


@server_mcp.resource("policy://acquisti-hardware", "Policy aziendale per gli acquisti hardware")
def policy_acquisti() -> str:
    return POLICY_TEXT


@server_mcp.prompt()
def valutazione_acquisto(employee_id: str, prodotto: str) -> str:
    """Template standard per valutare una richiesta di acquisto."""
    return (f"Valuta la richiesta del dipendente {employee_id} per il prodotto '{prodotto}'. "
            "Verifica budget e catalogo, applica la policy e indica APPROVATO, ESCALATION o RIFIUTO.")


print("Server MCP pronto. Tool registrati:", list(server_mcp._tools))

### Il client MCP

L'host non chiama mai direttamente le funzioni del server: passa da un **client** che parla solo JSON-RPC. In questo modo server e host possono essere scritti da persone diverse, in linguaggi diversi.

In [ ]:
class MiniMCPClient:
    """Client MCP: comunica col server esclusivamente tramite messaggi JSON-RPC."""

    def __init__(self, server: MiniMCPServer):
        self.server = server
        self._id = 0

    def _chiama(self, metodo: str, params: dict = None) -> dict:
        self._id += 1
        risposta = self.server.handle({"jsonrpc": "2.0", "id": self._id, "method": metodo, "params": params or {}})
        if "error" in risposta:
            raise RuntimeError(f"Errore MCP {risposta['error']['code']}: {risposta['error']['message']}")
        return risposta["result"]

    def list_tools(self) -> list:
        return self._chiama("tools/list")["tools"]

    def call_tool(self, nome: str, argomenti: dict) -> dict:
        esito = self._chiama("tools/call", {"name": nome, "arguments": argomenti})
        testo = esito["content"][0]["text"]
        return {"status": "ERROR", "message": testo} if esito["isError"] else json.loads(testo)

    def list_resources(self) -> list:
        return self._chiama("resources/list")["resources"]

    def read_resource(self, uri: str) -> str:
        return self._chiama("resources/read", {"uri": uri})["contents"][0]["text"]

    def list_prompts(self) -> list:
        return self._chiama("prompts/list")["prompts"]

    def get_prompt(self, nome: str, argomenti: dict) -> str:
        return self._chiama("prompts/get", {"name": nome, "arguments": argomenti})["messages"][0]["content"]["text"]


client_mcp = MiniMCPClient(server_mcp)


def tools_per_ollama(client: MiniMCPClient) -> list:
    """L'host traduce lo schema MCP (inputSchema) nel formato richiesto da Ollama (parameters)."""
    return [{"type": "function", "function": {"name": t["name"], "description": t["description"],
                                              "parameters": t["inputSchema"]}} for t in client.list_tools()]


print("Client MCP pronto.")

### ⚙️ Esploriamo le tre primitive

Interroghiamo il server come farebbe un'applicazione: cosa espone? Qual è lo schema dei tool? Cosa dice la policy?

In [ ]:
print("── RESOURCES ──")
for r in client_mcp.list_resources():
    print(" ", r["uri"], "-", r["description"])

print("\n── PROMPTS ──")
for p in client_mcp.list_prompts():
    print(" ", p["name"], [a["name"] for a in p["arguments"]])
print("  Esempio:", client_mcp.get_prompt("valutazione_acquisto", {"employee_id": "E4521", "prodotto": "ThinkPad"}))

print("\n── TOOLS (schema generato in automatico) ──")
for t in client_mcp.list_tools():
    print(f"  {t['name']}")
    print(f"     descrizione : {t['description'][:70]}...")
    print(f"     obbligatori : {t['inputSchema']['required']}")

print("\n── Lettura della resource policy (prime righe) ──")
print("\n".join(client_mcp.read_resource("policy://acquisti-hardware").splitlines()[:5]))

### ⚙️ Cosa viaggia davvero sul filo

Una singola `tools/call`, mostrata come richiesta e risposta JSON-RPC, più due casi di errore. Distingui:

- **errore di protocollo** (`error`): metodo o tool inesistente;
- **errore di esecuzione** (`isError: true`): il tool è stato chiamato ma è fallito.

In [ ]:
richiesta = {"jsonrpc": "2.0", "id": 42, "method": "tools/call",
             "params": {"name": "get_employee_budget", "arguments": {"employee_id": "E4521"}}}
print("RICHIESTA :", json.dumps(richiesta, ensure_ascii=False))
print("RISPOSTA  :", json.dumps(server_mcp.handle(richiesta), ensure_ascii=False)[:230], "...\n")

# Errore di PROTOCOLLO: metodo inesistente
print("Metodo sbagliato :", server_mcp.handle({"jsonrpc": "2.0", "id": 1, "method": "tools/inventato"}))

# Errore di ESECUZIONE: argomento mancante -> il tool fallisce ma il protocollo funziona
print("Argomento mancante:", server_mcp.handle({"jsonrpc": "2.0", "id": 2, "method": "tools/call",
                                                "params": {"name": "get_employee_budget", "arguments": {}}})["result"])

# Un 'errore di business' invece è un risultato normale del tool
print("Badge inesistente :", client_mcp.call_tool("get_employee_budget", {"employee_id": "E_NON_ESISTE"}))

### Il server vero, con l'SDK ufficiale

Con la libreria ufficiale `mcp` (`pip install mcp`) lo stesso server si scrive così. L'unica differenza è il decoratore: lo schema viene generato allo stesso modo.

```python
# simple_mcp_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("hardware-assistant")

@mcp.tool()
def get_employee_budget(employee_id: str) -> dict:
    """Restituisce budget residuo e ruolo di un dipendente dato il badge ID."""
    ...

@mcp.resource("policy://acquisti-hardware")
def policy() -> str:
    return open("data/policy_hardware.md", encoding="utf-8").read()

@mcp.prompt()
def valutazione_acquisto(employee_id: str, prodotto: str) -> str:
    return f"Valuta la richiesta di {employee_id} per {prodotto}."

if __name__ == "__main__":
    mcp.run(transport="stdio")      # oppure un trasporto HTTP
```

>NB) L'API dell'SDK evolve: prima di usarla in un progetto verifica la documentazione della versione installata.

### 🧠 Il loop agentico con tool MCP

Ora riusiamo `agent_loop` della Parte 1 **senza modificarlo**: gli diamo gli schemi presi dal server (`tools_per_ollama`) e `client_mcp.call_tool` come esecutore. La policy arriva dalla **resource** del server, non da un file letto dall'app.

In [ ]:
policy_da_mcp = client_mcp.read_resource("policy://acquisti-hardware")

SYSTEM_PROMPT_MCP = f"""Sei l'Assistente Acquisti Hardware aziendale.

=== POLICY AZIENDALE ===
{policy_da_mcp}
========================

REGOLE OPERATIVE:
1. Chiama SEMPRE 'get_employee_budget' e 'search_product' prima di rispondere.
2. Se prezzo <= budget residuo E rispetta la policy: l'acquisto è APPROVATO. NON chiamare 'request_manager_approval'.
3. Se prezzo > budget residuo OPPURE viola la policy (es. monitor > 300 EUR): chiama 'request_manager_approval'
   con employee_id, sku e motivazione, poi comunica il numero di ticket.
4. Rispondi in italiano."""

if OLLAMA_OK:
    TICKET_MCP.clear()
    print("═" * 60, "\nMario Rossi (E4521) — ThinkPad, atteso: APPROVATO senza ticket\n" + "═" * 60)
    agent_loop("Sono Mario Rossi (E4521). Posso acquistare il ThinkPad P16s per il Project Delta?",
               SYSTEM_PROMPT_MCP, tools_per_ollama(client_mcp), client_mcp.call_tool)

    print("\n" + "═" * 60, "\nGiulia Verdi (E1082) — monitor 4K, atteso: ticket al manager\n" + "═" * 60)
    agent_loop("Sono Giulia Verdi (E1082). Posso comprare il monitor 4K Dell UltraSharp?",
               SYSTEM_PROMPT_MCP, tools_per_ollama(client_mcp), client_mcp.call_tool)
    print("\nTicket creati nel server:", TICKET_MCP)
else:
    print(AVVISO_OLLAMA)

### Esercizio MCP — aggiungi un tool

**Consegna:** aggiungi al server un tool `check_order_status(order_id: str)` che, dato un ID ordine (es. `ORD-9876`), restituisce `in_transit` se l'ID inizia con `ORD-`, altrimenti `not_found`. Poi verifica che compaia in `tools/list` e provalo.

La cosa da notare: **non hai toccato né il client né il loop**. Il nuovo tool compare da solo, ed è proprio il vantaggio di un protocollo standard.

In [ ]:
# SOLUZIONE
@server_mcp.tool()
def check_order_status(order_id: str) -> dict:
    """Verifica lo stato di spedizione di un ordine esistente dato il suo ID (es. 'ORD-123')."""
    if order_id.startswith("ORD-"):
        return {"status": "in_transit", "estimated_delivery": "2 giorni lavorativi"}
    return {"status": "not_found", "message": "ID ordine non valido"}


print("Tool visibili ora:", [t["name"] for t in client_mcp.list_tools()])
print(client_mcp.call_tool("check_order_status", {"order_id": "ORD-9876"}))
print(client_mcp.call_tool("check_order_status", {"order_id": "XYZ-1"}))

## A2A — Agent2Agent

MCP collega un agente ai **suoi strumenti**. **A2A** collega **agenti autonomi tra loro**, anche di team o fornitori diversi: ad esempio il nostro agente interno che ordina hardware a un agente di un partner logistico.

La differenza chiave è la **trasparenza**:

| | **MCP** | **A2A** |
| :--- | :--- | :--- |
| Relazione | agente ➜ tool / dati | agente ➜ agente |
| Trasparenza | **trasparente**: l'agente vede ogni chiamata | **opaca**: l'agente remoto è una "scatola nera" |
| Ciclo di vita | chiamata di funzione, sincrona | **Task** asincrono con stati |
| Scopo | dare "mani e occhi" a un modello | far collaborare agenti indipendenti |

#### I quattro concetti di A2A

1. **Agent Card**: il "biglietto da visita" JSON pubblicato su `/.well-known/agent-card.json`, con nome, endpoint e **skill** offerte. Serve alla *discovery*.
2. **Task**: l'unità di lavoro, con una macchina a stati.
3. **Message e Parts**: i messaggi scambiati, composti da parti (`text`, `data`, `file`).
4. **Artifact**: il risultato finale e strutturato prodotto dall'agente remoto.

```
submitted ──► working ──► completed
                 │  ▲
                 ▼  │
           input-required        (l'agente remoto chiede un dato mancante)
                 
working ──► failed                (errore)
```

Il chiamante vede **solo** lo stato del task e gli artifact, non il ragionamento interno dell'altro agente.

In [ ]:
import uuid
from enum import Enum
from dataclasses import dataclass, field


class TaskState(str, Enum):
    SUBMITTED = "submitted"
    WORKING = "working"
    INPUT_REQUIRED = "input-required"
    COMPLETED = "completed"
    FAILED = "failed"


@dataclass
class A2ATask:
    task_id: str
    skill_id: str
    status: TaskState = TaskState.SUBMITTED
    history: list = field(default_factory=list)            # messaggi scambiati
    artifacts: list = field(default_factory=list)          # risultati finali
    state_transitions: list = field(default_factory=list)  # traccia degli stati
    context: dict = field(default_factory=dict)            # memoria interna dell'agente remoto

    def set_state(self, nuovo: TaskState, nota: str = ""):
        self.state_transitions.append({"from": self.status.value, "to": nuovo.value, "note": nota})
        self.status = nuovo

    def add_message(self, ruolo: str, parts: list):
        self.history.append({"role": ruolo, "parts": parts})


class BaseA2AAgent:
    """Agente A2A: pubblica una Agent Card, riceve task (lato server) e ne invia (lato client)."""

    def __init__(self, agent_id: str, name: str, description: str, endpoint: str, skills: list = None):
        self.agent_id, self.name, self.description = agent_id, name, description
        self.endpoint, self.skills = endpoint, skills or []
        self.tasks: dict[str, A2ATask] = {}

    def get_agent_card(self) -> dict:
        """Contenuto di /.well-known/agent-card.json (versione semplificata)."""
        return {"name": self.name, "description": self.description, "url": self.endpoint, "version": "1.0.0",
                "capabilities": {"streaming": False, "pushNotifications": False},
                "defaultInputModes": ["text/plain"], "defaultOutputModes": ["application/json"],
                "skills": self.skills}

    # ── Lato SERVER ──────────────────────────────────────────────────────
    def receive_task(self, skill_id: str, parts: list) -> A2ATask:
        task = A2ATask(task_id=uuid.uuid4().hex[:8], skill_id=skill_id)
        task.add_message("user", parts)
        self.tasks[task.task_id] = task
        self.process_task(task)
        return task

    def provide_input(self, task_id: str, parts: list) -> A2ATask:
        task = self.tasks[task_id]
        if task.status != TaskState.INPUT_REQUIRED:
            raise ValueError(f"Il task è in stato '{task.status.value}', non attende input.")
        task.add_message("user", parts)
        task.set_state(TaskState.WORKING, "input ricevuto, si riprende il lavoro")
        self.process_task(task)
        return task

    def process_task(self, task: A2ATask):
        raise NotImplementedError("Ogni agente implementa la propria logica.")

    @staticmethod
    def testo_utente(task: A2ATask) -> str:
        return " ".join(p.get("text", "") for m in task.history if m["role"] == "user"
                        for p in m["parts"] if p.get("kind") == "text")

    # ── Lato CLIENT ──────────────────────────────────────────────────────
    def send_task(self, target_agent: "BaseA2AAgent", skill_id: str, input_parts: list) -> A2ATask:
        skill_disponibili = [s["id"] for s in target_agent.get_agent_card()["skills"]]
        if skill_id not in skill_disponibili:
            raise ValueError(f"Skill '{skill_id}' non offerta da {target_agent.name}. Disponibili: {skill_disponibili}")
        return target_agent.receive_task(skill_id, input_parts)


print("Infrastruttura A2A definita.")

### L'agente fornitore e l'agente interno

L'**IT Procurement Agent** (del partner logistico) offre la skill `order_hardware_device`. Riceve uno SKU: se non conosce la sede di consegna non può procedere e porta il task in `input-required`. Quando riceve il dato, completa il task e produce un **Artifact** con l'ordine.

In [ ]:
class ProcurementITAgent(BaseA2AAgent):
    """Agente del fornitore esterno: emette ordini di hardware IT."""

    def __init__(self):
        super().__init__(
            agent_id="agent.partner.it-procurement", name="IT Procurement Agent",
            description="Agente del partner logistico che emette ordini di dispositivi hardware IT.",
            endpoint="https://partner.example/a2a/v1",
            skills=[{"id": "order_hardware_device", "name": "Ordina dispositivo hardware",
                     "description": "Emette un ordine per uno SKU del catalogo; richiede la sede di consegna.",
                     "tags": ["procurement", "hardware"],
                     "examples": ["Ordina il laptop LAP-DEV-01 per Mario Rossi"]}])
        self._n_ordini = 0

    def process_task(self, task: A2ATask):
        if task.status == TaskState.SUBMITTED:
            task.set_state(TaskState.WORKING, "analisi della richiesta")
            testo = self.testo_utente(task)
            trovato = re.search(r"[A-Z]{3}-[A-Z0-9]+-\d+", testo)
            if not trovato:
                task.add_message("agent", [{"kind": "text", "text": "Nessuno SKU riconosciuto nella richiesta."}])
                task.set_state(TaskState.FAILED, "SKU mancante")
                return
            task.context["sku"] = trovato.group(0)
            if "consegna" not in testo.lower():
                task.add_message("agent", [{"kind": "text", "text": "Indicare la sede di consegna del dispositivo."}])
                task.set_state(TaskState.INPUT_REQUIRED, "manca la sede di consegna")
                return
            task.context["sede"] = "indicata nella richiesta"
        elif task.status == TaskState.WORKING:           # ripresa dopo input-required
            task.context["sede"] = task.history[-1]["parts"][0]["text"]
        self._completa(task)

    def _completa(self, task: A2ATask):
        self._n_ordini += 1
        task.artifacts.append({
            "artifactId": uuid.uuid4().hex[:8], "name": "conferma-ordine",
            "parts": [{"kind": "data", "data": {"order_id": f"ORD-{self._n_ordini:04d}", "sku": task.context["sku"],
                                                "sede_consegna": task.context["sede"],
                                                "consegna_stimata": "3 giorni lavorativi"}}]})
        task.add_message("agent", [{"kind": "text", "text": "Ordine emesso."}])
        task.set_state(TaskState.COMPLETED, "ordine emesso")


class InternalCompanyAgent(BaseA2AAgent):
    """Agente interno: in questo scenario fa solo da CLIENT."""

    def __init__(self):
        super().__init__("agent.internal.assistant", "Internal Employee Assistant",
                         "Agente aziendale interno per le richieste dei dipendenti.", "https://internal.corp/a2a/v1")

    def process_task(self, task: A2ATask):
        pass


it_procurement_agent = ProcurementITAgent()
internal_agent = InternalCompanyAgent()
print("Agenti creati:", internal_agent.name, "e", it_procurement_agent.name)

### ⚙️ Passo 1 — Discovery

Prima di collaborare, il chiamante legge l'Agent Card per scoprire **quali skill** offre l'altro agente.

In [ ]:
agent_card = it_procurement_agent.get_agent_card()
print("AGENT CARD  (/.well-known/agent-card.json)")
print(json.dumps(agent_card, indent=2, ensure_ascii=False))

### ⚙️ Passo 2 — Invio del task e stato `input-required`

L'agente interno invia la richiesta **senza** indicare la sede. Osserva lo stato in cui si ferma il task.

In [ ]:
task = internal_agent.send_task(
    target_agent=it_procurement_agent,
    skill_id="order_hardware_device",
    input_parts=[{"kind": "text",
                  "text": "Richiesta emissione ordine per SKU: LAP-DEV-01 (Quantità: 1) per dipendente Mario Rossi (E4521)."}],
)

print("Task ID:", task.task_id)
print("Stato  :", task.status.value.upper())
print("Ultimo messaggio dell'agente remoto:", task.history[-1]["parts"][0]["text"])

### ⚙️ Passo 3 — Fornitura dell'input mancante e completamento

Il protocollo prevede un modo formale per riprendere un task sospeso (`provide_input`). Dopo l'input il task passa a `completed` e compare l'**Artifact**.

In [ ]:
it_procurement_agent.provide_input(
    task.task_id,
    [{"kind": "text", "text": "Milano HQ - Piano 3, Ufficio Tech."}],
)

print("Nuovo stato:", task.status.value.upper())
print("\nARTIFACT:")
print(json.dumps(task.artifacts, indent=2, ensure_ascii=False))

print("\nTRACCIA DEGLI STATI:")
for t in task.state_transitions:
    print(f"  • {t['from']:15s} ➜ {t['to']:15s} | {t['note']}")

# Un task già concluso non accetta altro input: errore gestito dal protocollo
try:
    it_procurement_agent.provide_input(task.task_id, [{"kind": "text", "text": "altro"}])
except ValueError as errore:
    print("\nErrore atteso:", errore)

### 🧠 L'LLM genera il messaggio A2A

Nella pratica l'utente scrive in linguaggio naturale e un LLM traduce la richiesta in un messaggio strutturato, scegliendo la skill giusta **leggendo l'Agent Card**. Chiediamo a Ollama di rispondere solo in JSON (`"format": "json"`).

In [ ]:
def genera_messaggio_a2a(richiesta_utente: str, card: dict) -> dict:
    """Usa l'LLM per scegliere la skill e scrivere il testo, in base all'Agent Card."""
    prompt = f"""Sei un assistente che prepara richieste A2A verso un agente fornitore.
Agent Card dell'agente destinatario:
{json.dumps(card, indent=2, ensure_ascii=False)}

Richiesta dell'utente: "{richiesta_utente}"

Rispondi ESCLUSIVAMENTE con un oggetto JSON valido:
{{"skill_id": "id_della_skill_scelta", "text_message": "testo dettagliato della richiesta"}}"""
    payload = {"model": MODEL_NAME, "messages": [{"role": "user", "content": prompt}],
               "stream": False, "format": "json", "options": {"temperature": 0}}
    risposta = requests.post(f"{OLLAMA_URL}/api/chat", json=payload, timeout=120)
    risposta.raise_for_status()
    return json.loads(risposta.json()["message"]["content"])


if OLLAMA_OK:
    messaggio_a2a = genera_messaggio_a2a("Ordina con urgenza il laptop LAP-DEV-01 per Mario Rossi", agent_card)
    print(json.dumps(messaggio_a2a, indent=2, ensure_ascii=False))
else:
    print(AVVISO_OLLAMA)

### Esercizio A2A — il `FinanceApprovalAgent`

**Consegna:** crea un agente con la skill `approve_budget_expense`.

1. Importo `<= 1000€` ➜ task `completed` con artifact `{"esito": "APPROVED"}`.
2. Importo `> 1000€` ➜ stato `input-required` con la richiesta *"Fornire il codice centro di costo (Cost Center) per approvazione manageriale"*.
3. Ricevuto il centro di costo ➜ il task si completa.

In [ ]:
# SOLUZIONE
class FinanceApprovalAgent(BaseA2AAgent):
    def __init__(self):
        super().__init__(
            agent_id="agent.finance.approval", name="Finance & Budget Approval Agent",
            description="Agente responsabile dell'autorizzazione spese e dei centri di costo.",
            endpoint="https://finance.corp/a2a/v1",
            skills=[{"id": "approve_budget_expense", "name": "Approva spesa",
                     "description": "Approva una spesa; sopra 1000€ richiede il centro di costo.",
                     "tags": ["finance"], "examples": ["Approva la spesa di 800 euro"]}])

    def process_task(self, task: A2ATask):
        if task.status == TaskState.SUBMITTED:
            task.set_state(TaskState.WORKING, "valutazione dell'importo")
            trovato = re.search(r"(\d+(?:[.,]\d+)?)\s*(?:€|eur|euro)", self.testo_utente(task), re.IGNORECASE)
            if not trovato:
                task.add_message("agent", [{"kind": "text", "text": "Importo non riconosciuto."}])
                task.set_state(TaskState.FAILED, "importo mancante")
                return
            importo = float(trovato.group(1).replace(",", "."))
            task.context["importo"] = importo
            if importo <= 1000:
                self._concludi(task, {"esito": "APPROVED", "importo": importo})
            else:
                task.add_message("agent", [{"kind": "text", "text":
                                            "Fornire il codice centro di costo (Cost Center) per approvazione manageriale"}])
                task.set_state(TaskState.INPUT_REQUIRED, "importo sopra soglia")
        elif task.status == TaskState.WORKING:            # ripresa con il centro di costo
            centro = task.history[-1]["parts"][0]["text"]
            self._concludi(task, {"esito": "APPROVED", "importo": task.context["importo"], "cost_center": centro})

    def _concludi(self, task: A2ATask, dati: dict):
        task.artifacts.append({"artifactId": uuid.uuid4().hex[:8], "name": "esito-approvazione",
                               "parts": [{"kind": "data", "data": dati}]})
        task.set_state(TaskState.COMPLETED, "spesa approvata")


finance = FinanceApprovalAgent()

piccola = internal_agent.send_task(finance, "approve_budget_expense",
                                   [{"kind": "text", "text": "Approva la spesa di 800 euro per il monitor."}])
print("Spesa 800€  ->", piccola.status.value, piccola.artifacts[0]["parts"][0]["data"])

grande = internal_agent.send_task(finance, "approve_budget_expense",
                                  [{"kind": "text", "text": "Approva la spesa di 2500 euro per 3 laptop."}])
print("Spesa 2500€ ->", grande.status.value, "|", grande.history[-1]["parts"][0]["text"])
finance.provide_input(grande.task_id, [{"kind": "text", "text": "CC-ENG-042"}])
print("Dopo l'input ->", grande.status.value, grande.artifacts[0]["parts"][0]["data"])

## Riepilogo Parte 2

- **MCP** standardizza il collegamento **agente ➜ tool/dati** con tre primitive: *Tools* (decide il modello), *Resources* (decide l'app), *Prompts* (decide l'utente). Messaggi JSON-RPC, schemi generati da type hint e docstring.
- **A2A** standardizza il collegamento **agente ➜ agente**: Agent Card per la discovery, Task con stati (`submitted`, `working`, `input-required`, `completed`, `failed`), Artifact come risultato.
- I due protocolli sono **complementari**: A2A tra agenti, MCP dentro ogni agente verso i suoi strumenti.
- Un protocollo ti fa risparmiare integrazioni, ma **non costruisce l'agente**: per quello servono i framework della Parte 3 e 4.

# Parte 3 — LangChain: costruire l'agente con un framework

Nella Parte 1 abbiamo scritto tutto a mano: schemi JSON, chiamate HTTP, loop ReAct. **LangChain** sostituisce questi pezzi con blocchi pronti.

| Cosa facevamo a mano (Parte 1) | Blocco LangChain |
| :--- | :--- |
| `requests.post` verso Ollama | `ChatOllama` |
| schema JSON scritto a mano | decoratore `@tool` (lo genera da docstring e type hint) |
| lista di messaggi con il system prompt | `ChatPromptTemplate` |
| `for` che gestisce il ciclo ReAct | `AgentExecutor` |
| `print` di ogni passo | `verbose=True` e `return_intermediate_steps=True` |

>NB) **Langflow** (l'editor visuale a blocchi) è costruito sopra LangChain: ogni componente visivo (Agent, Chat Input, Tool...) corrisponde a una classe o funzione LangChain. Scrivere in Python dà visibilità completa su ciò che Langflow fa internamente.

## Dal Project Canvas all'architettura

Il **Project Canvas** è una scheda che descrive *cosa* deve fare il sistema: caso d'uso, input, output, agenti, tool, contesto, guardrail. L'**architettura agentica** descrive *come* costruirlo con blocchi reali.

Il lavoro di un progettista è **tradurre ogni voce del canvas in un componente**:

| Voce del Canvas | Componente Langflow | Classe / funzione LangChain |
| :--- | :--- | :--- |
| Richiesta del dipendente | `Chat Input` | stringa passata come `input` |
| Agente principale | `Agent` | `create_tool_calling_agent()` + `AgentExecutor` |
| LLM | componente LLM | `ChatOllama(model="llama3.2")` |
| System prompt + policy | campo System Prompt | `ChatPromptTemplate` |
| Tool (budget, catalogo, approvazione) | `Custom Tool` | funzioni decorate con `@tool` |
| Guardrail / Human Approval | `Human Approval` | callback personalizzato (+ `interrupt` in LangGraph) |
| Agent-as-Tool | Agent in *Tool Mode* | `Tool.from_function()` che avvolge un secondo agente |
| Risposta finale | `Chat Output` | `risultato["output"]` |

In [ ]:
project_canvas = {
    "nome_progetto": "Assistente Aziendale Acquisto Hardware",
    "caso_duso": "Un dipendente invia una richiesta di acquisto. L'assistente verifica policy, budget e catalogo, "
                 "poi decide: approvazione automatica, escalation al manager o rifiuto.",
    "input": ["Richiesta in linguaggio naturale", "Badge ID dipendente"],
    "output": ["Approvazione automatica", "Escalation al manager", "Rifiuto motivato"],
    "agenti": [{"nome": "Assistente Acquisti", "ruolo": "Orchestra i tool e formula la decisione"}],
    "tools": [
        {"nome": "get_employee_budget", "descr": "Budget residuo e ruolo del dipendente"},
        {"nome": "search_product_catalog", "descr": "Cerca prodotti nel catalogo"},
        {"nome": "request_purchase_approval", "descr": "Crea un ticket di escalation al manager"},
    ],
    "contesto": "Policy aziendale (nel system prompt)",
    "guardrail": ["Human approval per spese oltre soglia", "Rifiuto se il prodotto non è a catalogo"],
    "mvp": "1 agente + 2 tool (budget + catalogo) + policy nel prompt",
}
print(json.dumps(project_canvas, indent=2, ensure_ascii=False))

## I quattro blocchi fondamentali

### 1. `ChatOllama` — il modello

Rappresenta il LLM. `temperature=0` rende le risposte il più possibile deterministiche, che è ciò che vogliamo in un agente che prende decisioni. Il costruttore non apre nessuna connessione: la chiamata parte solo con `.invoke()`.

In [ ]:
import datetime
from langchain_ollama import ChatOllama
from langchain_core.tools import tool, Tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.output_parsers import StrOutputParser

try:
    from langchain.agents import AgentExecutor, create_tool_calling_agent       # LangChain < 1.0
except ImportError:
    from langchain_classic.agents import AgentExecutor, create_tool_calling_agent  # LangChain >= 1.0

llm = ChatOllama(model=MODEL_NAME, temperature=0)
print("ChatOllama creato. Modello:", llm.model)

if OLLAMA_OK:
    risposta = llm.invoke([HumanMessage(content="Rispondi con una sola parola: qual è la capitale d'Italia?")])
    print("Risposta di test:", risposta.content)
else:
    print(AVVISO_OLLAMA)

### 2. `@tool` — gli strumenti

Il decoratore `@tool` trasforma una funzione Python in un tool. **Genera da solo lo schema JSON** leggendo type hint e docstring: niente più JSON scritto a mano.

>NB) **La docstring è lo schema.** È l'unica cosa che il modello legge per decidere *quando* chiamare il tool. Una docstring vaga produce un agente che usa male i tool; descrivi sempre **in quale situazione** serve.

In [ ]:
@tool
def esempio_tool(nome: str, valore: int) -> dict:
    """Restituisce un dizionario di esempio. Usare per dimostrare il funzionamento dei tool."""
    return {"nome": nome, "valore_doppio": valore * 2}


print("Nome        :", esempio_tool.name)
print("Descrizione :", esempio_tool.description)
print("Argomenti   :", esempio_tool.args)
print("\nSchema JSON completo:")
print(json.dumps(esempio_tool.args_schema.model_json_schema(), indent=2, ensure_ascii=False))

# Un tool si può chiamare direttamente, senza LLM: ottimo per testarlo
print("\nChiamata diretta:", esempio_tool.invoke({"nome": "ciao", "valore": 21}))

### 3. `ChatPromptTemplate` — il prompt

Il template definisce la struttura dei messaggi inviati al modello:

| Elemento | Ruolo |
| :--- | :--- |
| `("system", ...)` | ruolo dell'agente, regole, policy |
| `MessagesPlaceholder("chat_history", optional=True)` | spazio per la conversazione precedente |
| `("human", "{input}")` | la richiesta dell'utente (variabile `{input}`) |
| `MessagesPlaceholder("agent_scratchpad")` | **obbligatorio**: qui l'agente scrive tool call e risultati intermedi |

>NB) Nei template le graffe `{ }` sono variabili. Se nel prompt vuoi scrivere graffe letterali (per esempio un esempio di JSON) devi **raddoppiarle**: `{{ }}`. Dimenticarlo è uno degli errori più comuni.

In [ ]:
prompt_esempio = ChatPromptTemplate.from_messages([
    ("system", "Sei un assistente di esempio. Rispondi sempre in italiano."),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

print("Variabili di input:", prompt_esempio.input_variables)

### 4. `create_tool_calling_agent` + `AgentExecutor` — l'agente

Sono due pezzi con ruoli diversi:

- `create_tool_calling_agent(llm, tools, prompt)` crea il **ragionatore**: sa chiedere i tool al modello;
- `AgentExecutor` gestisce il **loop ReAct**: esegue i tool richiesti, rimanda i risultati, si ferma alla risposta finale.

```python
agent    = create_tool_calling_agent(llm, tools, prompt)   # il ragionatore
executor = AgentExecutor(agent=agent, tools=tools)         # il loop
risultato = executor.invoke({"input": "..."})              # esecuzione
```

In [ ]:
agent_esempio = create_tool_calling_agent(llm, [esempio_tool], prompt_esempio)
executor_esempio = AgentExecutor(agent=agent_esempio, tools=[esempio_tool], verbose=True)

if OLLAMA_OK:
    risultato = executor_esempio.invoke({"input": "Usa il tool con nome=ciao e valore=21"})
    print("\nOutput finale:", risultato["output"])
else:
    print(AVVISO_OLLAMA)

## L'MVP: un agente con due tool

**MVP** (*Minimum Viable Product*) è la versione più piccola che funziona davvero. Per il nostro caso: **1 agente + 2 tool di sola lettura** (budget e catalogo) + la policy nel prompt.

L'obiettivo è rispondere correttamente a **3 richieste con esiti diversi**: approvazione automatica, escalation, rifiuto.

### Passo 1 — I tool

Riscriviamo i tool con `@tool`. Le docstring dicono **quando** chiamarli.

In [ ]:
@tool
def get_employee_budget(employee_id: str) -> dict:
    """Recupera budget residuo, ruolo, dipartimento e progetto attivo di un dipendente.
    Chiamare SEMPRE prima di valutare una richiesta, per conoscere budget e fascia di spesa.
    Argomento: employee_id, il BADGE ID alfanumerico (es. 'E4521'), NON il nome."""
    emp = EMPLOYEES_DB.get(employee_id)
    if emp is None:
        return {"status": "NOT_FOUND", "message": f"Dipendente '{employee_id}' non trovato."}
    return {"status": "OK", "employee_id": employee_id, "name": emp["name"], "department": emp["department"],
            "role": emp["role"], "budget_residuo_eur": emp["budget_residuo"],
            "active_project": emp["active_project"], "manager_email": emp["manager_email"]}


@tool
def search_product_catalog(query: str) -> dict:
    """Cerca prodotti nel catalogo hardware aziendale per nome o categoria.
    Usare SEMPRE per verificare se il prodotto esiste a catalogo e conoscerne il prezzo ufficiale.
    Categorie: laptop, monitor, accessory. Esempi di query: 'ThinkPad', 'monitor', 'Dell'."""
    trovati = cerca_prodotti(query)
    if not trovati:
        return {"status": "NOT_FOUND", "message": f"Nessun prodotto per '{query}'. Prodotto NON presente a catalogo."}
    return {"status": "OK", "risultati": trovati, "totale": len(trovati)}


tools_mvp = [get_employee_budget, search_product_catalog]

# ⚙️ I tool si testano da soli, senza LLM
print(get_employee_budget.invoke({"employee_id": "E1082"}))
print(search_product_catalog.invoke({"query": "Apple Vision Pro"}))

### Passo 2 — Il system prompt

Un buon prompt per un agente ha sempre quattro elementi: **ruolo**, **sequenza operativa** (quali tool, in che ordine), **esiti possibili** con le condizioni per ciascuno, **lingua** della risposta. La policy aziendale viene iniettata come testo.

In [ ]:
SYSTEM_PROMPT_MVP = f"""Sei l'Assistente Aziendale per gli Acquisti Hardware.

Sequenza operativa obbligatoria prima di rispondere:
1. Chiama 'get_employee_budget' con il badge ID del dipendente (es. 'E4521').
2. Chiama SEMPRE 'search_product_catalog' per verificare che il prodotto sia a catalogo e leggerne il prezzo esatto.
3. Decidi con queste regole:
   - RIFIUTO: dipendente non trovato, oppure prodotto NON a catalogo (NOT_FOUND). Non inventare mai prezzi o modelli.
   - APPROVAZIONE AUTOMATICA: prodotto a catalogo, rientra nella fascia del ruolo ed è coperto dal budget residuo.
   - ESCALATION AL MANAGER: prodotto a catalogo ma oltre la fascia/soglia della policy, oppure prezzo superiore al budget residuo.
4. Rispondi SEMPRE in italiano, indicando la decisione (APPROVAZIONE AUTOMATICA, ESCALATION AL MANAGER o RIFIUTO)
   e la motivazione con i numeri.

=== POLICY AZIENDALE ===
{POLICY_TEXT}
========================
"""

prompt_mvp = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT_MVP),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

agent_mvp = create_tool_calling_agent(llm, tools_mvp, prompt_mvp)
executor_mvp = AgentExecutor(
    agent=agent_mvp,
    tools=tools_mvp,
    verbose=True,                 # mostra ogni passo del ragionamento
    max_iterations=8,             # limite di sicurezza contro i loop infiniti
    handle_parsing_errors=True,   # non crashare se il modello produce un output malformato
)
print("Executor MVP pronto. Tool:", [t.name for t in tools_mvp])

### Passo 3 — I tre scenari 🧠

| Scenario | Richiesta | Esito atteso |
| :--- | :--- | :--- |
| 1 | Mario Rossi (E4521) vuole il ThinkPad P16s a 1650€, budget 1800€ | ✅ approvazione automatica |
| 2 | Giulia Verdi (E1082) vuole il monitor 4K a 520€, budget 450€ | ⚠️ escalation al manager |
| 3 | Alessandro Corti (E9900) vuole un Apple Vision Pro | ❌ rifiuto: non a catalogo |

In [ ]:
SCENARI_MVP = [
    {"label": "Scenario 1 — Approvazione automatica",
     "input": "Ciao! Sono Mario Rossi, badge E4521. Lavoro su Project Delta e ho bisogno di un laptop performante. "
              "Posso acquistare il ThinkPad P16s? Rientra nella policy?",
     "atteso": ["APPROVAZIONE AUTOMATICA", "APPROVATO"]},
    {"label": "Scenario 2 — Escalation al manager",
     "input": "Salve, sono Giulia Verdi (badge E1082) del team Marketing. Per la Brand Campaign Q2 ho bisogno "
              "del monitor 4K Dell UltraSharp. Posso procedere all'acquisto?",
     "atteso": ["ESCALATION"]},
    {"label": "Scenario 3 — Rifiuto, prodotto non a catalogo",
     "input": "Ciao, sono Alessandro Corti, badge E9900. Ho bisogno di un Apple Vision Pro per le presentazioni ai clienti. "
              "Come posso procedere?",
     "atteso": ["RIFIUTO", "RIFIUTATO"]},
]

risposte_mvp = []
if OLLAMA_OK:
    for scenario in SCENARI_MVP:
        print("\n" + "═" * 60 + f"\n  {scenario['label']}\n" + "═" * 60)
        risultato = executor_mvp.invoke({"input": scenario["input"]})
        risposte_mvp.append(risultato["output"])
        print("\n📤 RISPOSTA FINALE:", risultato["output"])
else:
    print(AVVISO_OLLAMA)

In [ ]:
# Verifica automatica: la risposta contiene la parola chiave dell'esito atteso?
if risposte_mvp:
    tutti_ok = True
    for scenario, risposta in zip(SCENARI_MVP, risposte_mvp):
        ok = any(k.lower() in risposta.lower() for k in scenario["atteso"])
        tutti_ok &= ok
        print(f"{'✅' if ok else '⚠️ '} {scenario['label']}")
    print("\n🏆 Obiettivo raggiunto!" if tutti_ok else "\n⚠️  Controlla a mano le risposte sopra.")
    print("\nSe il modello sbaglia: 1) migliora la docstring del tool (più specifica sul QUANDO), "
          "2) rendi esplicita la sequenza nel prompt, 3) ricorda 'Rispondi SEMPRE in italiano'.")
else:
    print("Nessuna risposta da verificare (celle 🧠 non eseguite).")

### Passo 4 — Cosa fa `AgentExecutor` dentro?

Con `return_intermediate_steps=True` otteniamo la lista dei passi eseguiti, come coppie *(azione, osservazione)*: è il modo per **ispezionare** e **testare** il ragionamento, senza leggere i log a occhio.

In [ ]:
executor_silenzioso = AgentExecutor(agent=agent_mvp, tools=tools_mvp, verbose=False,
                                    max_iterations=8, return_intermediate_steps=True)

if OLLAMA_OK:
    dettaglio = executor_silenzioso.invoke({"input": "Ciao, sono Mario Rossi (E4521). Voglio il ThinkPad P16s."})
    print("Chiavi dell'output:", list(dettaglio.keys()))
    print("Risposta:", dettaglio["output"][:200], "\n")
    for i, (azione, osservazione) in enumerate(dettaglio["intermediate_steps"], 1):
        print(f"Passo {i}: tool='{azione.tool}'  args={azione.tool_input}")
        print(f"         risultato={str(osservazione)[:110]}")
else:
    print(AVVISO_OLLAMA)

### Esercizi

**A.** Aggiungi un quarto scenario: Alessandro Corti (E9900) chiede il *Thunderbolt 4 Dock* (190€). Atteso: approvazione automatica (accessorio sotto i 300€, budget 600€).

**B.** Copia il tool `get_employee_budget` e **togli dalla docstring** l'istruzione "Chiamare SEMPRE prima di...". Riesegui lo scenario 1. Il modello lo chiama ancora? Cosa cambia? (Serve a capire quanto pesa la docstring.)

In [ ]:
# SOLUZIONE ESERCIZIO A
if OLLAMA_OK:
    r = executor_silenzioso.invoke({"input": "Ciao, sono Alessandro Corti (E9900). Vorrei il Thunderbolt 4 Dock. Posso?"})
    print(r["output"])
else:
    print(AVVISO_OLLAMA)

## Pattern Human Approval

Un agente autonomo non deve decidere tutto da solo. Quando la decisione è **ad alto impatto** (spese oltre soglia, azioni irreversibili, dati sensibili) deve fermarsi e chiedere conferma a una persona.

Il pattern ha tre pezzi:

1. un **tool di escalation** (`request_purchase_approval`) che crea un ticket;
2. un **checkpoint umano** che mostra il riepilogo e chiede conferma;
3. un **audit trail**: il registro di chi ha deciso cosa e quando.

In LangChain l'ingresso naturale per osservare ciò che fa l'agente sono i **callback** (`BaseCallbackHandler`):

| Evento | Quando scatta |
| :--- | :--- |
| `on_tool_start` | prima dell'esecuzione di un tool |
| `on_tool_end` | dopo l'esecuzione di un tool |
| `on_agent_finish` | quando l'agente ha la risposta finale |

#### Scelta di design: la logica deterministica sta nel tool

Con un modello piccolo, ogni chiamata in più è un'occasione di errore. Invece di lasciare che l'LLM concateni quattro tool, esponiamo **un solo tool** `evaluate_purchase_request`, che dentro (in Python) legge i dati, applica `applica_policy` e crea il ticket se serve. L'LLM deve solo estrarre badge e prodotto e comunicare l'esito.

>NB) **Attenzione a dove si registra il callback.** Un callback passato al *costruttore* di `AgentExecutor` (`AgentExecutor(..., callbacks=[cb])`) riceve gli eventi dell'esecutore ma **non quelli dei tool**. Per ricevere `on_tool_start` e `on_tool_end` va passato **al momento dell'invocazione**: `executor.invoke(input, config={"callbacks": [cb]})`.

In [ ]:
TICKET_STORE: list[dict] = []   # "database" dei ticket (in memoria)


@tool
def request_purchase_approval(employee_id: str, product_name: str, price_eur: float, reason: str) -> dict:
    """Crea un ticket di approvazione per il manager. Usare per richieste che superano budget, fascia o soglia."""
    ticket = {
        "ticket_id": f"TKT-{len(TICKET_STORE) + 1:04d}",
        "employee_id": employee_id,
        "employee_name": EMPLOYEES_DB.get(employee_id, {}).get("name", employee_id),
        "manager_email": EMPLOYEES_DB.get(employee_id, {}).get("manager_email", "manager@azienda.it"),
        "product": product_name,
        "price_eur": float(price_eur),
        "reason": reason,
        "status": "PENDING_APPROVAL",
        "created_at": datetime.datetime.now().isoformat(),
    }
    TICKET_STORE.append(ticket)
    return {"status": "PENDING_HUMAN_APPROVAL", "ticket_id": ticket["ticket_id"],
            "message": f"Richiesta registrata con ID {ticket['ticket_id']}. In attesa del manager."}


@tool
def evaluate_purchase_request(employee_id: str, product_query: str) -> dict:
    """Valuta una richiesta di acquisto: legge i dati del dipendente, cerca il prodotto a catalogo,
    applica la policy e, se serve, crea il ticket per il manager. Restituisce la decisione finale.
    employee_id: BADGE ID alfanumerico (es. 'E1082'), NON il nome. product_query: nome o categoria del prodotto."""
    emp = EMPLOYEES_DB.get(employee_id)
    if emp is None:
        return {"decision": "REJECTED", "reason": f"Dipendente '{employee_id}' non trovato."}

    trovati = cerca_prodotti(product_query)
    if not trovati:
        return {"decision": "REJECTED", "reason": f"Prodotto '{product_query}' non presente a catalogo."}

    prod = next(iter(trovati.values()))
    esito, motivo = applica_policy(emp, prod)

    if esito == "ESCALATION":
        # Si chiama la funzione Python sottostante (.func), non il tool: non è una nuova tool call dell'LLM
        ticket = request_purchase_approval.func(employee_id=employee_id, product_name=prod["name"],
                                                price_eur=prod["price_eur"], reason=motivo)
        return {"decision": "NEEDS_HUMAN_APPROVAL", "ticket_id": ticket["ticket_id"], "product": prod["name"],
                "price_eur": prod["price_eur"], "reason": motivo}

    return {"decision": "APPROVED", "employee": emp["name"], "product": prod["name"],
            "price_eur": prod["price_eur"], "reason": motivo}


print("Tool definiti:", request_purchase_approval.name, "e", evaluate_purchase_request.name)

### ⚙️ Testiamo il tool senza LLM

Tutta la decisione è in Python, quindi la possiamo verificare in modo esatto, scenario per scenario.

In [ ]:
casi = [
    ("E4521", "ThinkPad",                  "APPROVED"),
    ("E1082", "monitor 4K Dell UltraSharp", "NEEDS_HUMAN_APPROVAL"),
    ("E9900", "Apple Vision Pro",          "REJECTED"),
    ("E9999", "laptop",                    "REJECTED"),
]
for badge, query, atteso in casi:
    esito = evaluate_purchase_request.invoke({"employee_id": badge, "product_query": query})
    assert esito["decision"] == atteso, esito
    print(f"{badge} / {query:28s} -> {esito['decision']:22s} {esito.get('reason', '')}")

print("\nTicket creati:", [(t["ticket_id"], t["product"], t["status"]) for t in TICKET_STORE])
TICKET_STORE.clear()

### Callback e checkpoint umano

- `HumanApprovalCallback` osserva i tool: quando `evaluate_purchase_request` risponde `NEEDS_HUMAN_APPROVAL` scrive un evento nell'**audit log**;
- `human_approval_checkpoint()` cerca i ticket `PENDING_APPROVAL`, mostra il riepilogo e chiede la decisione all'operatore.

>NB) In Jupyter un `input()` dentro un callback non viene instradato alla cella. Per questo il checkpoint interattivo è una **funzione chiamata nella cella**, dopo la fine dell'agente. In produzione si usa invece `interrupt()` di LangGraph (Parte 4), che sospende *davvero* il flusso e lo riprende dopo la risposta umana.

In [ ]:
from typing import Any


class HumanApprovalCallback(BaseCallbackHandler):
    """Osserva le tool call e registra un evento nell'audit log quando nasce un ticket."""

    def __init__(self):
        self.audit_log = []

    def on_tool_end(self, output: Any, **kwargs: Any) -> None:
        if "NEEDS_HUMAN_APPROVAL" in str(output):
            ticket = TICKET_STORE[-1] if TICKET_STORE else {}
            self.audit_log.append({"event": "ticket_creato", "ticket_id": ticket.get("ticket_id"),
                                   "timestamp": datetime.datetime.now().isoformat()})


def human_approval_checkpoint(approval_cb: HumanApprovalCallback, auto_mode: bool = True, auto_decisione: bool = True):
    """Chiede (o simula) l'approvazione umana per ogni ticket in attesa e la registra nell'audit trail.
    auto_mode=True evita input() e usa auto_decisione: comodo per 'Esegui tutto'. Metti False per l'uso interattivo."""
    in_attesa = [t for t in TICKET_STORE if t["status"] == "PENDING_APPROVAL"]
    if not in_attesa:
        print("Nessuna escalation: niente da approvare.")
        return

    for ticket in in_attesa:
        print("\n" + "═" * 60 + "\n  🔐 HUMAN APPROVAL CHECKPOINT\n" + "═" * 60)
        print(f"  Dipendente : {ticket['employee_name']} ({ticket['employee_id']})")
        print(f"  Prodotto   : {ticket['product']} — {ticket['price_eur']} EUR")
        print(f"  Motivazione: {ticket['reason']}")
        print(f"  Manager    : {ticket['manager_email']}   Ticket: {ticket['ticket_id']}")

        if auto_mode:
            approvato, nota = auto_decisione, "Decisione simulata (auto_mode)"
            print(f"  [AUTO_MODE] decisione simulata: {'APPROVATO' if approvato else 'RIFIUTATO'}")
        else:
            approvato = input("\n  Approvi il ticket? [s/n]: ").strip().lower() in ("s", "si", "y", "yes")
            nota = input("  Nota (invio per saltare): ").strip() or ("Approvato" if approvato else "Rifiutato")

        ticket["status"] = "APPROVED" if approvato else "REJECTED_BY_OPERATOR"
        ticket["operator_note"] = nota
        approval_cb.audit_log.append({"event": "decisione_umana", "ticket_id": ticket["ticket_id"],
                                      "approved": approvato, "note": nota,
                                      "timestamp": datetime.datetime.now().isoformat()})
        print(f"  📁 Audit salvato: {ticket['ticket_id']} -> {'APPROVATO' if approvato else 'RIFIUTATO'}")


# ⚙️ Prova del callback e del checkpoint SENZA LLM: simuliamo a mano la sequenza di eventi
cb_prova = HumanApprovalCallback()
risultato_tool = evaluate_purchase_request.invoke({"employee_id": "E1082", "product_query": "monitor 4K"})
cb_prova.on_tool_end(str(risultato_tool))          # è ciò che farebbe LangChain al termine del tool
human_approval_checkpoint(cb_prova, auto_mode=True, auto_decisione=False)   # il manager RIFIUTA
print("\nAudit log:", json.dumps(cb_prova.audit_log, indent=2, ensure_ascii=False))
TICKET_STORE.clear()

### 🧠 Il flow completo con l'agente

Il prompt V2 dice all'agente di usare **solo** `evaluate_purchase_request`. Il callback viene passato **in `config` all'invocazione**, come spiegato nell'avviso sopra.

In [ ]:
SYSTEM_PROMPT_V2 = """Sei l'Assistente Aziendale per gli Acquisti Hardware.
Hai un solo tool: 'evaluate_purchase_request'. Chiamalo SEMPRE con il badge ID del dipendente (es. 'E1082')
e il nome del prodotto. Il tool applica la policy, crea il ticket se serve e restituisce la decisione.
Dopo la risposta del tool comunica l'esito all'utente in italiano (APPROVATO, IN ATTESA DEL MANAGER o RIFIUTATO)."""

prompt_v2 = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT_V2),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

tools_v2 = [evaluate_purchase_request]
executor_v2 = AgentExecutor(agent=create_tool_calling_agent(llm, tools_v2, prompt_v2), tools=tools_v2,
                            verbose=False, max_iterations=6, handle_parsing_errors=True,
                            return_intermediate_steps=True)

if OLLAMA_OK:
    TICKET_STORE.clear()
    approval_cb = HumanApprovalCallback()

    print("═" * 60, "\n  Giulia Verdi: escalation + approvazione umana\n" + "═" * 60)
    res_giulia = executor_v2.invoke(
        {"input": "Salve, sono Giulia Verdi (badge E1082), Marketing. Ho bisogno del monitor 4K Dell UltraSharp. Posso?"},
        config={"callbacks": [approval_cb]},            # <-- qui, NON nel costruttore
    )
    print("📤 AGENTE:", res_giulia["output"])
    human_approval_checkpoint(approval_cb, auto_mode=True, auto_decisione=True)   # auto_mode=False per rispondere tu

    print("\n" + "═" * 60, "\n  Mario Rossi: approvazione automatica, il checkpoint NON si attiva\n" + "═" * 60)
    res_mario = executor_v2.invoke(
        {"input": "Ciao, sono Mario Rossi, badge E4521. Mi serve il ThinkPad P16s per Project Delta."},
        config={"callbacks": [approval_cb]},
    )
    print("📤 AGENTE:", res_mario["output"])
else:
    print(AVVISO_OLLAMA)

In [ ]:
# Audit trail e passaggi intermedi
if OLLAMA_OK:
    print("AUDIT LOG:")
    for record in approval_cb.audit_log:
        print(" ", json.dumps(record, ensure_ascii=False))
    print("\nTICKET:")
    for t in TICKET_STORE:
        print(f"  [{t['ticket_id']}] {t['status']} — {t['employee_name']} / {t['product']}")
    print("\nPASSAGGI INTERMEDI (Giulia):")
    for azione, osservazione in res_giulia["intermediate_steps"]:
        print(f"  tool={azione.tool}  args={azione.tool_input}")
        print(f"  ->   {str(osservazione)[:140]}")
else:
    print(AVVISO_OLLAMA)

### Esercizio — un secondo callback: `ToolLogger`

**Consegna:** scrivi un callback che registra **nome, durata e orario** di ogni tool call senza interrompere il flusso, e passalo insieme a `HumanApprovalCallback`. Un callback di osservabilità come questo è la base del monitoraggio in produzione.

In [ ]:
# SOLUZIONE
import time


class ToolLogger(BaseCallbackHandler):
    def __init__(self):
        self.log = []
        self._inizio = 0.0
        self._tool = ""

    def on_tool_start(self, serialized: dict, input_str: str, **kwargs: Any) -> None:
        self._inizio = time.time()
        self._tool = kwargs.get("name") or (serialized or {}).get("name", "sconosciuto")

    def on_tool_end(self, output: Any, **kwargs: Any) -> None:
        self.log.append({"tool": self._tool, "durata_s": round(time.time() - self._inizio, 4),
                         "timestamp": datetime.datetime.now().isoformat()})

    def report(self):
        print("📊 Log di esecuzione dei tool:")
        for voce in self.log:
            print(f"  {voce['tool']:28s} {voce['durata_s']:.4f}s")


# ⚙️ Prova senza LLM: simuliamo start/end
logger = ToolLogger()
logger.on_tool_start({"name": "evaluate_purchase_request"}, "{}")
evaluate_purchase_request.invoke({"employee_id": "E4521", "product_query": "ThinkPad"})
logger.on_tool_end("ok")
logger.report()

# Con l'agente:  executor_v2.invoke({...}, config={"callbacks": [approval_cb, logger]})

## Pattern Agent-as-Tool

Quando un unico agente deve conoscere policy, catalogo, budget e flusso, il suo prompt diventa enorme e il modello si confonde. La soluzione è **dividere le competenze**: sotto-agenti specializzati, ognuno con un prompt piccolo e pochi tool, e un **orchestratore** che sa *quando* delegare.

Il trucco è `Tool.from_function()`: avvolge qualsiasi funzione Python (anche una che invoca un altro agente) in un tool standard. L'orchestratore lo vede come un tool qualunque e non sa che dietro c'è un altro agente.

```
input ──► Orchestratore (AgentExecutor)
            ├─ @tool  get_employee_budget
            ├─ Tool.from_function ──► Sotto-agente Policy   (solo prompt con la policy, nessun tool)
            └─ Tool.from_function ──► Sotto-agente Catalogo (solo il tool search_product_catalog)
```

>NB) Un sotto-agente **senza tool** è semplicemente una *chain*: `prompt | llm | parser`. Non serve un `AgentExecutor`. Quello del Catalogo, che ha un tool, sì.

### Passo 1 — I sotto-agenti

In [ ]:
# ── Sotto-agente POLICY: nessun tool, solo un prompt con la policy ───────
prompt_policy = ChatPromptTemplate.from_messages([
    ("system", f"""Sei lo specialista della Policy Aziendale Acquisti Hardware.

=== POLICY ===
{POLICY_TEXT}
==============

Quando ricevi una richiesta esegui SEMPRE due controlli:
 A) Fascia/soglia: il prezzo rispetta il limite per ruolo/categoria (laptop: 1800€ Engineering, 1000€ altri; monitor/accessori: 300€)?
 B) Budget: il prezzo è <= al budget residuo del dipendente?
Decisione: APPROVAZIONE AUTOMATICA se A e B sono veri; ESCALATION AL MANAGER se uno dei due è falso.
Limite di fascia e budget residuo sono valori DIVERSI: confrontali separatamente.
Rispondi in italiano mostrando i numeri dei due controlli e la decisione."""),
    ("human", "{input}"),
])
policy_chain = prompt_policy | llm | StrOutputParser()


# ── Sotto-agente CATALOGO: un solo tool, nessuna conoscenza di policy ────
prompt_catalogo = ChatPromptTemplate.from_messages([
    ("system", """Sei lo specialista del catalogo hardware aziendale.
Il tuo UNICO compito è cercare prodotti e riportare nome, prezzo, categoria e ruoli target.
REGOLA ASSOLUTA: chiama SEMPRE 'search_product_catalog' prima di rispondere; il catalogo è l'unica fonte di verità.
Estrai UNA parola chiave breve dalla domanda (es. 'laptop', 'monitor', 'ThinkPad'). Rispondi in italiano.
NON hai conoscenza di policy, budget o dipendenti."""),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])
catalogo_executor = AgentExecutor(
    agent=create_tool_calling_agent(llm, [search_product_catalog], prompt_catalogo),
    tools=[search_product_catalog], verbose=False, handle_parsing_errors=True,   # verbose=False: i log interni sarebbero rumore
)
print("Sotto-agenti pronti: policy_chain, catalogo_executor")

### Passo 2 — Avvolgerli come tool

La **descrizione** del tool è il contratto con l'orchestratore: deve dire *quando* delegare e *che tipo di domanda* passare.

In [ ]:
agente_policy_tool = Tool.from_function(
    func=lambda domanda: policy_chain.invoke({"input": domanda}),
    name="agente_verifica_policy",
    description=("Sotto-agente specialista della policy acquisti hardware. Usare per verificare se una combinazione "
                 "(ruolo, prodotto, prezzo, budget residuo) è in policy o richiede approvazione. "
                 "Passare una frase completa, es.: 'Dipendente Marketing, budget 450€, vuole un monitor da 520€.'"),
)

agente_catalogo_tool = Tool.from_function(
    func=lambda domanda: catalogo_executor.invoke({"input": domanda})["output"],
    name="agente_catalogo",
    description=("Sotto-agente specialista del catalogo hardware. Usare per cercare prodotti, prezzi e disponibilità. "
                 "Passare una domanda sul prodotto, es.: 'Quanto costa il ThinkPad P16s?'"),
)

print("Tool creati:", agente_policy_tool.name, "|", agente_catalogo_tool.name)

### Passo 3 — L'orchestratore

L'orchestratore **non conosce la policy**: la delega. Ha una sequenza obbligatoria e una regola d'oro: *non inventare mai prezzi o budget, usa solo i valori restituiti dai tool*.

In [ ]:
prompt_orchestratore = ChatPromptTemplate.from_messages([
    ("system", """Sei l'Assistente Aziendale per gli Acquisti Hardware — ORCHESTRATORE.

Tool a disposizione:
- 'get_employee_budget': budget, ruolo e progetto del dipendente
- 'agente_catalogo': cerca prodotti, prezzi e disponibilità
- 'agente_verifica_policy': verifica se una richiesta è in policy

Sequenza OBBLIGATORIA:
1. Chiama 'get_employee_budget' con il BADGE ID (lettera + cifre, es. 'E4521'), non con il nome. Annota ruolo, dipartimento e budget.
2. Chiama 'agente_catalogo' con il nome del prodotto. Annota il prezzo ESATTO restituito.
3. Chiama 'agente_verifica_policy' con UNA frase che contiene ruolo, prodotto, prezzo esatto e budget residuo.
   Esempio: 'Dipendente Senior Developer (Engineering), budget residuo 1800€, vuole il ThinkPad P16s a 1650€.'
4. Decidi basandoti SOLO sui dati raccolti e rispondi in italiano: APPROVAZIONE AUTOMATICA, ESCALATION AL MANAGER o RIFIUTO, con motivazione.

CRITICO: non inventare prezzi, budget o ruoli. Non devi conoscere la policy: delegala sempre allo specialista."""),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

tools_orchestratore = [get_employee_budget, agente_policy_tool, agente_catalogo_tool]
orchestratore = AgentExecutor(
    agent=create_tool_calling_agent(llm, tools_orchestratore, prompt_orchestratore),
    tools=tools_orchestratore, verbose=False, max_iterations=12,
    handle_parsing_errors=True, return_intermediate_steps=True,
)
print("Orchestratore pronto. Tool:", [t.name for t in tools_orchestratore])

In [ ]:
RICHIESTE_AAT = [
    ("Mario Rossi — ThinkPad (atteso: approvazione)",
     "Ciao! Sono Mario Rossi, badge E4521. Posso acquistare il ThinkPad P16s per il Project Delta?"),
    ("Giulia Verdi — monitor 4K (atteso: escalation)",
     "Salve, sono Giulia Verdi (badge E1082), Marketing. Posso acquistare il monitor 4K Dell UltraSharp?"),
]

if OLLAMA_OK:
    for etichetta, richiesta in RICHIESTE_AAT:
        print("\n" + "═" * 60 + f"\n  {etichetta}\n" + "═" * 60)
        r = orchestratore.invoke({"input": richiesta})
        print("📤", r["output"])
        print("\n  Catena di chiamate:")
        for azione, osservazione in r["intermediate_steps"]:
            tipo = "🤖 sotto-agente" if azione.tool.startswith("agente_") else "🔧 tool"
            print(f"   {tipo:16s} {azione.tool:24s} in: {str(azione.tool_input)[:70]}")
else:
    print(AVVISO_OLLAMA)

### Monolitico o Agent-as-Tool?

| | Agente monolitico (MVP) | Agent-as-Tool |
| :--- | :--- | :--- |
| Struttura | 1 agente, 1 prompt con tutto | orchestratore + specialisti |
| Contesto del modello | grande, rischio di confusione tra domini | piccolo e mirato per ogni agente |
| Chiamate LLM | poche | **molte** (più latenza e costo) |
| Manutenzione | cambi la policy = tocchi il prompt grande | cambi **un solo** sotto-agente |
| Errori | meno punti di rottura | più punti di rottura, ma più isolati |
| Quando usarlo | dominio piccolo, MVP | domini complessi con conoscenze separate che cambiano spesso |

Il pattern Agent-as-Tool è il **ponte verso i sistemi multi-agente** della Parte 4, dove la collaborazione diventa un grafo con stato esplicito.

### Esercizio — un terzo sotto-agente

**Consegna:** crea `agente_budget_analyzer`, che usa `get_employee_budget` e classifica la capacità di acquisto: **ALTA** (> 1500€), **MEDIA** (500–1500€), **BASSA** (< 500€). Avvolgilo con `Tool.from_function()` e aggiungilo all'orchestratore.

In [ ]:
# SOLUZIONE
prompt_budget = ChatPromptTemplate.from_messages([
    ("system", """Sei lo specialista di capacità di acquisto. Chiama SEMPRE 'get_employee_budget' con il badge ID
(es. 'E4521') e classifica: ALTA se budget > 1500€, MEDIA se tra 500€ e 1500€, BASSA se < 500€.
Rispondi in italiano con il budget esatto e la classificazione."""),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])
budget_executor = AgentExecutor(
    agent=create_tool_calling_agent(llm, [get_employee_budget], prompt_budget),
    tools=[get_employee_budget], verbose=False, handle_parsing_errors=True,
)
agente_budget_tool = Tool.from_function(
    func=lambda domanda: budget_executor.invoke({"input": domanda})["output"],
    name="agente_budget_analyzer",
    description="Sotto-agente che classifica la capacità di acquisto (ALTA/MEDIA/BASSA) di un dipendente dato il badge ID, es. 'E1082'.",
)
tools_orchestratore_v2 = tools_orchestratore + [agente_budget_tool]
print("Tool dell'orchestratore v2:", [t.name for t in tools_orchestratore_v2])
# Per usarlo: ricostruisci l'orchestratore con tools_orchestratore_v2 e cita il nuovo tool nel prompt.

## Esercitazione: il tuo MVP con LangChain

Ora applica tutto a un **tuo caso d'uso**. Il metodo è sempre lo stesso, in sei fasi:

| Fase | Cosa fai | Cosa produci |
| :--- | :--- | :--- |
| 1 — Dati | definisci i dati del dominio (dizionari o file in `data/`) | dataset + testo delle regole |
| 2 — Tool | scrivi almeno **2** funzioni `@tool` con docstring chiara | tool testabili con `.invoke()` |
| 3 — Prompt | ruolo + esiti elencati + sequenza dei tool + "rispondi in italiano" | `ChatPromptTemplate` |
| 4 — Executor | `create_tool_calling_agent` + `AgentExecutor` | l'agente |
| 5 — Test | **3 scenari con esiti diversi** | verifica automatica |
| 6 — Estensioni | Human Approval, terzo tool, Agent-as-Tool | (opzionale) |

Come **esempio svolto** vediamo lo scenario alternativo: un assistente HR per **ferie e permessi**.

| Esito | Regola |
| :--- | :--- |
| ✅ Approvazione automatica | permesso ≤ 4h con ore disponibili, o ferie con giorni sufficienti e buon preavviso |
| ⚠️ Escalation al manager | ferie o permessi ≥ 8h, oppure preavviso breve (< 5 giorni lavorativi) |
| ❌ Rifiuto | ferie residue insufficienti |

In [ ]:
# ── FASE 1: dati ─────────────────────────────────────────────────────────
DIPENDENTI_HR = {
    "HR001": {"nome": "Luca Ferrari",  "team": "Engineering", "ferie_residue": 18, "permessi_ore": 24},
    "HR002": {"nome": "Sara Martini",  "team": "Marketing",   "ferie_residue": 5,  "permessi_ore": 4},
    "HR003": {"nome": "Pietro Romano", "team": "Sales",       "ferie_residue": 22, "permessi_ore": 16},
}

POLICY_FERIE = """## Policy Ferie e Permessi
- Permessi orari brevi (<= 4h): approvazione automatica se le ore sono disponibili.
- Ferie o permessi >= 8h: richiedono la conferma del manager (escalation).
- Ferie residue inferiori ai giorni richiesti: rifiuto automatico.
- Preavviso inferiore a 5 giorni lavorativi: escalation al manager (salvo emergenze).
"""


# ── FASE 2: tool ─────────────────────────────────────────────────────────
@tool
def get_dipendente_hr(employee_id: str) -> dict:
    """Recupera i dati HR di un dipendente: ferie residue, ore di permesso, team.
    Chiamare SEMPRE per prima, prima di valutare qualsiasi richiesta. Badge es. 'HR001'."""
    if employee_id not in DIPENDENTI_HR:
        return {"status": "NOT_FOUND", "message": f"Dipendente '{employee_id}' non trovato."}
    return {"status": "OK", "employee_id": employee_id, **DIPENDENTI_HR[employee_id]}


@tool
def verifica_disponibilita_ferie(employee_id: str, giorni_richiesti: int) -> dict:
    """Verifica se il dipendente ha ferie residue sufficienti. Chiamare per le richieste di FERIE (non di permessi orari).
    Parametri: employee_id (badge) e giorni_richiesti (numero intero)."""
    if employee_id not in DIPENDENTI_HR:
        return {"status": "NOT_FOUND"}
    residue = DIPENDENTI_HR[employee_id]["ferie_residue"]
    return {"status": "OK", "ferie_residue": residue, "giorni_richiesti": giorni_richiesti,
            "disponibile": residue >= giorni_richiesti, "ferie_dopo": max(0, residue - giorni_richiesti)}


# ⚙️ test dei tool
print(get_dipendente_hr.invoke({"employee_id": "HR002"}))
print(verifica_disponibilita_ferie.invoke({"employee_id": "HR002", "giorni_richiesti": 10}))

In [ ]:
# ── FASE 3 e 4: prompt ed executor ───────────────────────────────────────
prompt_hr = ChatPromptTemplate.from_messages([
    ("system", f"""Sei l'Assistente HR per la gestione di ferie e permessi.

Esiti possibili:
- ✅ APPROVAZIONE AUTOMATICA: permesso <= 4h con ore disponibili, oppure ferie con giorni sufficienti e preavviso adeguato.
- ⚠️  ESCALATION AL MANAGER: ferie con preavviso breve, oppure permessi >= 8h.
- ❌ RIFIUTO: ferie residue insufficienti.

Sequenza:
1. Chiama 'get_dipendente_hr' per ferie residue e ore di permesso.
2. Se la richiesta è di FERIE chiama 'verifica_disponibilita_ferie'.
3. Confronta con la policy e decidi.
4. Rispondi in italiano con la decisione motivata.

=== POLICY ===
{POLICY_FERIE}
==============
"""),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

tools_hr = [get_dipendente_hr, verifica_disponibilita_ferie]
executor_hr = AgentExecutor(agent=create_tool_calling_agent(llm, tools_hr, prompt_hr), tools=tools_hr,
                            verbose=False, max_iterations=8, handle_parsing_errors=True)

# ── FASE 5: tre scenari con esiti diversi ────────────────────────────────
TEST_HR = [
    ("Caso 1 — permesso breve (atteso: approvazione)",
     "Ciao, sono Luca Ferrari (HR001). Devo assentarmi 3 ore domani per una visita. Posso?", ["APPROVAZIONE", "APPROVATO"]),
    ("Caso 2 — ferie insufficienti (atteso: rifiuto)",
     "Salve, sono Sara Martini (HR002). Vorrei 10 giorni di ferie in agosto. È possibile?", ["RIFIUTO", "RIFIUTATO"]),
    ("Caso 3 — preavviso breve (atteso: escalation)",
     "Ciao, sono Pietro Romano (HR003). Ho un'emergenza: mi servono 3 giorni di ferie da domani.", ["ESCALATION", "MANAGER"]),
]

if OLLAMA_OK:
    for etichetta, richiesta, atteso in TEST_HR:
        r = executor_hr.invoke({"input": richiesta})
        ok = any(k.lower() in r["output"].lower() for k in atteso)
        print(f"\n{'✅' if ok else '⚠️ '} {etichetta}\n   {r['output'][:300]}")
else:
    print(AVVISO_OLLAMA)

#### Lo scheletro per il tuo caso d'uso

Copia la cella seguente, sostituisci i `TODO` e rispondi mentalmente a queste domande prima di scrivere codice: *quali dati servono? quali sono gli esiti possibili? quali regole li distinguono?*

In [ ]:
# ── SCHELETRO: il tuo MVP ────────────────────────────────────────────────
MIO_DATASET = {"U001": {"nome": "TODO", "attributo_chiave": "TODO"}}
MIE_REGOLE = """## Regole del dominio
- TODO: regola 1
- TODO: regola 2
"""


@tool
def mio_tool_1(identificativo: str) -> dict:
    """TODO: scrivi QUANDO questo tool deve essere chiamato (in quale situazione servono questi dati?)."""
    return {"status": "OK", "dati": MIO_DATASET.get(identificativo)}


@tool
def mio_tool_2(parametro: str) -> dict:
    """TODO: scrivi QUANDO questo tool deve essere chiamato."""
    return {"status": "OK", "risultato": f"TODO per {parametro}"}


MIO_PROMPT = ChatPromptTemplate.from_messages([
    ("system", f"""TODO: Sei [ruolo specifico].
Esiti possibili: ✅ TODO / ⚠️ ESCALATION TODO / ❌ TODO.
Sequenza: 1. chiama 'mio_tool_1'  2. chiama 'mio_tool_2'  3. decidi.
Rispondi SEMPRE in italiano.

=== REGOLE ===
{MIE_REGOLE}
==============="""),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

MIO_TOOLS = [mio_tool_1, mio_tool_2]
mio_executor = AgentExecutor(agent=create_tool_calling_agent(llm, MIO_TOOLS, MIO_PROMPT), tools=MIO_TOOLS,
                             verbose=True, max_iterations=8, handle_parsing_errors=True)
print("Scheletro pronto: sostituisci i TODO e prova con mio_executor.invoke({'input': '...'})")

## Riepilogo Parte 3

- **`ChatOllama`**, **`@tool`**, **`ChatPromptTemplate`**, **`AgentExecutor`** sostituiscono schemi a mano, chiamate HTTP e loop ReAct.
- La **docstring** del tool è il suo schema: scrivila come se dovessi spiegare a un collega *quando* usarlo.
- **Human Approval**: tool di escalation + checkpoint umano + audit trail. I callback vanno passati in `config={"callbacks": [...]}` all'invocazione per vedere gli eventi dei tool.
- **Agent-as-Tool** (`Tool.from_function`): sotto-agenti specializzati orchestrati da un agente che sa *quando delegare*, a costo di più chiamate LLM.
- Le regole di business stanno nel **codice**, testabili con `.invoke()` senza LLM; il modello è usato dove serve il linguaggio.

Il limite di questi agenti è che il flusso lo decide il modello ad ogni passo e lo **stato è implicito**. Per un processo aziendale con esiti precisi servono più controllo e uno stato esplicito: è il compito di **LangGraph**.

# Parte 4 — LangGraph: agenti con stato, rami e cicli

## Perché LangGraph

Con `AgentExecutor` il flusso lo decide il modello a ogni passo e lo stato è implicito. Per un **processo aziendale** (validare, controllare, decidere, chiedere approvazione, riprovare se fallisce) serve poter dire *esattamente* cosa succede dopo cosa.

| | LangChain `AgentExecutor` | **LangGraph** |
| :--- | :--- | :--- |
| Stato | implicito (cronologia dei messaggi) | **esplicito** (uno schema che definisci tu) |
| Flusso | lo decide il LLM | lo decidi tu con archi e router (il LLM può decidere dentro un nodo) |
| Cicli e retry | difficili | naturali (un arco che torna indietro) |
| Pause per l'umano | solo con trucchi | **nativo** (`interrupt`) |
| Memoria tra chiamate | da aggiungere | **checkpointer** integrato |

Un programma LangGraph è un **grafo** fatto di tre cose:

| Elemento | Cos'è | In Python |
| :--- | :--- | :--- |
| **Stato** (*State*) | i dati condivisi da tutto il grafo | una classe `TypedDict` |
| **Nodi** (*Nodes*) | i passi di lavoro | funzioni `stato -> dict con i campi aggiornati` |
| **Archi** (*Edges*) | il collegamento tra i passi | `add_edge` (fisso) e `add_conditional_edges` (dipende dallo stato) |

Due nodi speciali delimitano il grafo: `START` e `END`.

#### I reducer: come si fonde l'output di un nodo nello stato

Un nodo restituisce solo i campi che ha cambiato. Cosa succede al campo già presente? Dipende dal **reducer**:

| Dichiarazione nello stato | Comportamento | Quando usarla |
| :--- | :--- | :--- |
| `contatore: int` | il nuovo valore **sostituisce** il vecchio | valori singoli |
| `log: Annotated[list[str], operator.add]` | il nuovo valore viene **accodato** | log, messaggi, risultati accumulati |
| `messages: Annotated[list, add_messages]` | accoda, ma **sostituisce** i messaggi con lo stesso `id` | conversazioni (serve per modificare un messaggio) |

In [ ]:
import operator
import uuid
from typing import TypedDict, Annotated

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, AIMessage, ToolMessage


def mostra_grafo(app, png: bool = False):
    """Mostra la struttura del grafo. Di default stampa il codice Mermaid (incollabile su mermaid.live).
    Con png=True prova a disegnarlo (usa un servizio esterno, serve Internet)."""
    if png:
        try:
            from IPython.display import Image, display
            display(Image(app.get_graph().draw_mermaid_png()))
            return
        except Exception as errore:
            print("PNG non disponibile:", errore)
    print(app.get_graph().draw_mermaid())


print("LangGraph pronto.")

## Primo grafo (senza LLM)

Cominciamo con un grafo giocattolo, per vedere **solo** i meccanismi: stato, nodi, archi, reducer. Nessun modello coinvolto.

Il campo `contatore` viene **sovrascritto**, il campo `log` viene **accumulato**.

In [ ]:
class StatoGiocattolo(TypedDict):
    contatore: int                           # nessun reducer: sovrascritto
    log: Annotated[list[str], operator.add]  # reducer 'add': accodato


def nodo_a(stato: StatoGiocattolo) -> dict:
    return {"contatore": stato["contatore"] + 1, "log": ["A eseguito"]}      # dict PARZIALE


def nodo_b(stato: StatoGiocattolo) -> dict:
    return {"contatore": stato["contatore"] * 10, "log": ["B eseguito"]}


costruttore = StateGraph(StatoGiocattolo)
costruttore.add_node("a", nodo_a)
costruttore.add_node("b", nodo_b)
costruttore.add_edge(START, "a")
costruttore.add_edge("a", "b")
costruttore.add_edge("b", END)

app_giocattolo = costruttore.compile()      # .compile() valida il grafo (nodi orfani, archi mancanti...)

risultato = app_giocattolo.invoke({"contatore": 1, "log": []})
print("Risultato:", risultato)
assert risultato == {"contatore": 20, "log": ["A eseguito", "B eseguito"]}   # (1+1)*10 = 20

print("\nStreaming: lo stato aggiornato da ciascun nodo")
for passo in app_giocattolo.stream({"contatore": 1, "log": []}):
    print("  ", passo)

print("\nStruttura (Mermaid):")
mostra_grafo(app_giocattolo)

## Archi condizionali e cicli

Un **arco condizionale** usa una funzione *router*: legge lo stato e restituisce il **nome del prossimo nodo**. Se un arco punta a un nodo già visitato otteniamo un **ciclo**, la base di retry e conversazioni.

>NB) Ogni ciclo ha bisogno di una **condizione di uscita** certa. LangGraph ha anche una rete di sicurezza: il `recursion_limit` (25 passi di default) blocca i cicli infiniti sollevando `GraphRecursionError`.

In [ ]:
from langgraph.errors import GraphRecursionError


class StatoCiclo(TypedDict):
    valore: int
    tentativi: int


def incrementa(stato: StatoCiclo) -> dict:
    return {"valore": stato["valore"] + 3, "tentativi": stato["tentativi"] + 1}


def router_ciclo(stato: StatoCiclo) -> str:
    """Il router restituisce la CHIAVE da cercare nella mappa degli archi condizionali."""
    return "fine" if stato["valore"] >= 10 else "ripeti"


g = StateGraph(StatoCiclo)
g.add_node("incrementa", incrementa)
g.add_edge(START, "incrementa")
g.add_conditional_edges("incrementa", router_ciclo, {"ripeti": "incrementa", "fine": END})   # chiave -> nodo
app_ciclo = g.compile()

risultato = app_ciclo.invoke({"valore": 0, "tentativi": 0})
print("Ciclo concluso:", risultato)          # 3, 6, 9, 12 -> 4 giri
assert risultato["tentativi"] == 4

try:
    app_ciclo.invoke({"valore": 0, "tentativi": 0}, config={"recursion_limit": 3})
except GraphRecursionError:
    print("Con recursion_limit=3 il ciclo viene interrotto: GraphRecursionError")

## Il grafo di orchestrazione degli acquisti

Ora il caso guida. Il flusso, con esiti e ciclo di chiarimento:

```
START ─► estrai_parametri ─► verifica_budget ─► cerca_catalogo ─► decide ──► END   (APPROVATO / ESCALATION / RIFIUTO)
              ▲                                                     │
              │                                                     └─ AMBIGUO ─► chiedi_chiarimento
              └─────────────────────────────────────────────────────────────────────────┘   (max 3 giri)
```

Principio di design già visto: **le regole nel codice, il linguaggio al modello**. I nodi che decidono sono deterministici; il LLM, se disponibile, scrive solo il messaggio finale per il dipendente (con un testo di ripiego se Ollama non c'è).

### Passo 1 — Lo stato

Ogni nodo scrive **il proprio campo** (`budget_data`, `catalog_data`, `esito`...). I chiarimenti dell'utente si accumulano con `operator.add`.

In [ ]:
class StatoRichiesta(TypedDict, total=False):       # total=False: i campi possono mancare all'inizio
    # Input
    richiesta: str
    chiarimenti: Annotated[list[str], operator.add]   # risposte dell'utente, accumulate
    employee_id: str
    prodotto_query: str
    # Risultati intermedi: ogni nodo scrive il suo
    budget_data: dict
    catalog_data: dict
    # Decisione
    esito: str            # APPROVATO | ESCALATION | RIFIUTO | AMBIGUO
    motivazione: str
    risposta_finale: str
    # Controllo del flusso
    retry_count: int


MAX_RETRY = 3
print("Campi dello stato:", list(StatoRichiesta.__annotations__))

### Passo 2 — I nodi

Ogni nodo è una funzione `stato -> dict parziale`. Nota che `decide` **non** chiede al LLM di confrontare numeri: usa `applica_policy`. Il LLM entra solo in `genera_testo`, che ha un ripiego se Ollama non è disponibile.

In [ ]:
def genera_testo(prompt: str, ripiego: str) -> str:
    """Usa il LLM per scrivere un testo; se Ollama non c'è (o fallisce) restituisce il testo di ripiego."""
    if not OLLAMA_OK:
        return ripiego
    try:
        return llm.invoke([HumanMessage(content=prompt)]).content.strip()
    except Exception:
        return ripiego


def estrai_parametri(stato: StatoRichiesta) -> dict:
    """Nodo 1: ricava badge ID e prodotto dal testo (richiesta + eventuali chiarimenti)."""
    print("  🔍 [estrai_parametri]")
    testo = " ".join([stato["richiesta"]] + stato.get("chiarimenti", []))
    badge = re.search(r"\bE\d{3,5}\b", testo)
    employee_id = stato.get("employee_id") or (badge.group(0) if badge else "UNKNOWN")
    # Parole del testo che compaiono nel catalogo (nome o categoria)
    prodotto_query = stato.get("prodotto_query") or " ".join(t for t in tokenizza(testo) if cerca_prodotti(t))
    print(f"     employee_id={employee_id!r}  prodotto_query={prodotto_query!r}")
    return {"employee_id": employee_id, "prodotto_query": prodotto_query}


def verifica_budget(stato: StatoRichiesta) -> dict:
    """Nodo 2: legge i dati del dipendente."""
    print("  💰 [verifica_budget]")
    emp = EMPLOYEES_DB.get(stato["employee_id"])
    if emp is None:
        return {"budget_data": {"status": "NOT_FOUND", "message": f"Dipendente '{stato['employee_id']}' non trovato."}}
    return {"budget_data": {"status": "OK", "employee_id": stato["employee_id"], **emp}}


def cerca_catalogo(stato: StatoRichiesta) -> dict:
    """Nodo 3: cerca il prodotto a catalogo."""
    print("  📦 [cerca_catalogo]")
    query = stato.get("prodotto_query", "")
    if not query:
        return {"catalog_data": {"status": "NO_QUERY"}}
    trovati = cerca_prodotti(query)
    if not trovati:
        return {"catalog_data": {"status": "NOT_FOUND", "message": f"Il prodotto '{query}' non è a catalogo."}}
    return {"catalog_data": {"status": "OK", "risultati": trovati}}


def decide(stato: StatoRichiesta) -> dict:
    """Nodo 4: decisione deterministica; il LLM (se c'è) scrive solo il messaggio."""
    print("  🧠 [decide]")
    budget, catalogo = stato["budget_data"], stato["catalog_data"]
    nome_prodotto = ""

    if budget["status"] != "OK":                              # failure path: dipendente sconosciuto
        esito, motivo = "RIFIUTO", budget["message"]
    elif catalogo["status"] == "NO_QUERY":                    # mancano informazioni
        esito, motivo = "AMBIGUO", "la richiesta non indica quale prodotto serve"
    elif catalogo["status"] == "NOT_FOUND":                   # failure path: prodotto sconosciuto
        esito, motivo = "RIFIUTO", catalogo["message"]
    else:
        prodotto = next(iter(catalogo["risultati"].values()))
        nome_prodotto = prodotto["name"]
        esito, motivo = applica_policy(budget, prodotto)      # budget contiene anche i dati del dipendente

    ripiego = {"APPROVATO": f"✅ APPROVATO: {nome_prodotto} ({motivo}).",
               "ESCALATION": f"⚠️ ESCALATION: {nome_prodotto}, serve il manager ({motivo}).",
               "RIFIUTO": f"❌ RIFIUTO: {motivo}",
               "AMBIGUO": f"❓ Servono più informazioni: {motivo}."}[esito]
    if esito == "AMBIGUO":
        testo = ripiego
    else:
        testo = genera_testo(f"Scrivi in italiano, in massimo due frasi, la risposta per un dipendente. "
                             f"Esito: {esito}. Prodotto: {nome_prodotto or 'n/d'}. Motivazione: {motivo}. "
                             f"Non cambiare l'esito.", ripiego)
    print(f"     -> {esito}")
    return {"esito": esito, "motivazione": motivo, "risposta_finale": testo}


RISPOSTE_UTENTE_SIMULATE = ["Mi serve un laptop ThinkPad per Project Delta."]


def chiedi_chiarimento(stato: StatoRichiesta) -> dict:
    """Nodo del ciclo: in un sistema vero aspetterebbe l'utente; qui la risposta è simulata."""
    giro = stato.get("retry_count", 0)
    risposta = RISPOSTE_UTENTE_SIMULATE[giro] if giro < len(RISPOSTE_UTENTE_SIMULATE) else "Non lo so."
    print(f"  ❓ [chiedi_chiarimento] giro {giro + 1}/{MAX_RETRY}  utente (simulato): {risposta!r}")
    return {"chiarimenti": [risposta], "retry_count": giro + 1}


print("Nodi definiti.")

### Passo 3 — Il router e la costruzione del grafo

Il router decide se uscire (`END`) o fare un altro giro di chiarimento. Il **limite di giri** (`MAX_RETRY`) è la condizione di uscita certa del ciclo.

>NB) I nomi dei nodi **non possono coincidere** con i nomi dei campi dello stato (LangGraph solleva un errore).

In [ ]:
def router_dopo_decide(stato: StatoRichiesta) -> str:
    if stato["esito"] == "AMBIGUO" and stato.get("retry_count", 0) < MAX_RETRY:
        return "chiedi_chiarimento"
    return END          # esiti finali, oppure troppi giri: terminazione forzata


costruttore = StateGraph(StatoRichiesta)
costruttore.add_node("estrai_parametri", estrai_parametri)
costruttore.add_node("verifica_budget", verifica_budget)
costruttore.add_node("cerca_catalogo", cerca_catalogo)
costruttore.add_node("decide", decide)
costruttore.add_node("chiedi_chiarimento", chiedi_chiarimento)

# Archi fissi: sequenza lineare
costruttore.add_edge(START, "estrai_parametri")
costruttore.add_edge("estrai_parametri", "verifica_budget")
costruttore.add_edge("verifica_budget", "cerca_catalogo")
costruttore.add_edge("cerca_catalogo", "decide")

# Arco condizionale: dopo 'decide' si esce oppure si chiede un chiarimento
costruttore.add_conditional_edges("decide", router_dopo_decide,
                                  {"chiedi_chiarimento": "chiedi_chiarimento", END: END})

# Ciclo: dopo il chiarimento si riparte dall'estrazione (il testo ora contiene la nuova risposta)
costruttore.add_edge("chiedi_chiarimento", "estrai_parametri")

app = costruttore.compile()
mostra_grafo(app)

### Passo 4 — Eseguiamo i cinque scenari

| # | Richiesta | Percorso | Esito atteso |
| :--- | :--- | :--- | :--- |
| 1 | Mario Rossi vuole il ThinkPad | lineare | `APPROVATO` |
| 2 | Giulia Verdi vuole il monitor 4K | lineare | `ESCALATION` |
| 3 | badge `E9999` inesistente | failure path | `RIFIUTO` |
| 4 | Alessandro Corti vuole un prodotto non a catalogo | failure path | `RIFIUTO` |
| 5 | richiesta vaga, poi chiarita | **ciclo** (1 giro) | `APPROVATO` |
| 6 | richiesta vaga, utente non collabora | **ciclo fino al limite** | `AMBIGUO` (terminazione forzata) |

Le `assert` verificano gli esiti: **funzionano anche senza Ollama**.

In [ ]:
def esegui(richiesta: str, employee_id: str = None, prodotto: str = None) -> dict:
    stato = {"richiesta": richiesta, "chiarimenti": [], "retry_count": 0}
    if employee_id:
        stato["employee_id"] = employee_id
    if prodotto:
        stato["prodotto_query"] = prodotto
    print("\n" + "═" * 62 + f"\n  🚀 {richiesta}\n" + "═" * 62)
    risultato = app.invoke(stato)
    print(f"\n  ESITO: {risultato['esito']}   (giri di chiarimento: {risultato.get('retry_count', 0)})")
    print(f"  {risultato['risposta_finale']}")
    return risultato


s1 = esegui("Ciao, sono Mario Rossi (E4521). Voglio acquistare il ThinkPad P16s.")
assert s1["esito"] == "APPROVATO"

s2 = esegui("Salve, sono Giulia Verdi (E1082). Ho bisogno del monitor 4K Dell UltraSharp.")
assert s2["esito"] == "ESCALATION"

s3 = esegui("Ciao, sono Giovanni Bianchi (E9999). Voglio un laptop.")
assert s3["esito"] == "RIFIUTO"

s4 = esegui("Ciao, sono Alessandro Corti (E9900). Mi serve un Apple Vision Pro.", prodotto="apple vision pro")
assert s4["esito"] == "RIFIUTO"

s5 = esegui("Sono Mario Rossi (E4521), mi serve qualcosa per lavorare meglio.")
assert s5["esito"] == "APPROVATO" and s5["retry_count"] == 1

print("\n✅ Scenari 1-5 corretti.")

In [ ]:
# Scenario 6: l'utente risponde sempre in modo vago -> il ciclo si ferma al limite di giri
RISPOSTE_UTENTE_SIMULATE[:] = ["Non lo so.", "Boh.", "Fate voi."]

s6 = esegui("Sono Mario Rossi (E4521), mi serve qualcosa per lavorare meglio.")
assert s6["esito"] == "AMBIGUO" and s6["retry_count"] == MAX_RETRY
print("\n✅ Scenario 6: terminazione forzata dopo", MAX_RETRY, "giri (failure path del ciclo).")

RISPOSTE_UTENTE_SIMULATE[:] = ["Mi serve un laptop ThinkPad per Project Delta."]   # ripristino

### Streaming: lo stato dopo ogni nodo

`.stream()` restituisce, uno alla volta, i dizionari **parziali** che ogni nodo ha prodotto. È lo strumento principale per fare debug di un grafo.

In [ ]:
print("Streaming passo-passo (richiesta con un giro di chiarimento):\n")
for passo in app.stream({"richiesta": "Sono Mario Rossi (E4521), mi serve qualcosa per lavorare meglio.",
                         "chiarimenti": [], "retry_count": 0}):
    for nodo, delta in passo.items():
        campi = {k: (json.dumps(v, ensure_ascii=False)[:60] if isinstance(v, (dict, list)) else v)
                 for k, v in delta.items() if k != "risposta_finale"}
        print(f"  [{nodo}] {campi}")

### Esercizio — un nodo di validazione

**Consegna:** aggiungi `valida_input` tra `START` e `estrai_parametri`: se la richiesta è vuota o più corta di 10 caratteri, imposta `esito = "ERRORE"` e termina. Servono un nodo, una funzione router e due archi.

In [ ]:
# SOLUZIONE
def valida_input(stato: StatoRichiesta) -> dict:
    testo = stato.get("richiesta", "").strip()
    if len(testo) < 10:
        return {"esito": "ERRORE", "motivazione": "richiesta vuota o troppo breve",
                "risposta_finale": "❌ Richiesta non valida: scrivi qualche dettaglio in più."}
    return {}      # nessun campo da aggiornare


def router_dopo_validazione(stato: StatoRichiesta) -> str:
    return END if stato.get("esito") == "ERRORE" else "estrai_parametri"


c = StateGraph(StatoRichiesta)
c.add_node("valida_input", valida_input)
for nome, funzione in [("estrai_parametri", estrai_parametri), ("verifica_budget", verifica_budget),
                       ("cerca_catalogo", cerca_catalogo), ("decide", decide), ("chiedi_chiarimento", chiedi_chiarimento)]:
    c.add_node(nome, funzione)
c.add_edge(START, "valida_input")
c.add_conditional_edges("valida_input", router_dopo_validazione, {END: END, "estrai_parametri": "estrai_parametri"})
c.add_edge("estrai_parametri", "verifica_budget")
c.add_edge("verifica_budget", "cerca_catalogo")
c.add_edge("cerca_catalogo", "decide")
c.add_conditional_edges("decide", router_dopo_decide, {"chiedi_chiarimento": "chiedi_chiarimento", END: END})
c.add_edge("chiedi_chiarimento", "estrai_parametri")
app_validato = c.compile()

print(app_validato.invoke({"richiesta": "ciao", "chiarimenti": [], "retry_count": 0})["risposta_finale"])
print(app_validato.invoke({"richiesta": "Sono Mario Rossi (E4521), voglio il ThinkPad.", "chiarimenti": [],
                           "retry_count": 0})["esito"])

## L'agente ReAct come grafo

Il loop ReAct della Parte 1 è esso stesso un grafo di **due nodi** e un arco condizionale:

```
START ─► llm ──(ha chiesto tool?)──sì──► action ──┐
          ▲                                       │
          └───────────────────────────────────────┘
          │
          no
          ▼
         END
```

Lo stato qui è **solo una lista di messaggi** (con il reducer `add_messages`): il LLM decide quali tool chiamare. Incapsuliamo il grafo in una classe `Agent` riutilizzabile: basta passare modello, tool e system prompt diversi. Accetta anche un **checkpointer** e `interrupt_before`, che useremo nelle sezioni successive.

>NB) Usiamo `add_messages` e non `operator.add`: come `add`, accoda i messaggi, ma **sostituisce quelli con lo stesso `id`**. È ciò che permette di *modificare* una tool call già pianificata con `update_state` (più avanti).

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]


class Agent:
    """Agente ReAct come grafo LangGraph: llm <-> action."""

    def __init__(self, model, tools, system: str = "", checkpointer=None, interrupt_before=None):
        self.system = system
        grafo = StateGraph(AgentState)
        grafo.add_node("llm", self.call_llm)
        grafo.add_node("action", self.take_action)
        grafo.add_edge(START, "llm")
        grafo.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        grafo.add_edge("action", "llm")
        self.graph = grafo.compile(checkpointer=checkpointer, interrupt_before=interrupt_before)
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)     # il modello "conosce" gli schemi dei tool

    def exists_action(self, stato: AgentState) -> bool:
        """Arco condizionale: il modello ha richiesto almeno un tool?"""
        return len(getattr(stato["messages"][-1], "tool_calls", [])) > 0

    def call_llm(self, stato: AgentState) -> dict:
        messaggi = stato["messages"]
        if self.system:
            messaggi = [SystemMessage(content=self.system)] + messaggi
        return {"messages": [self.model.invoke(messaggi)]}

    def take_action(self, stato: AgentState) -> dict:
        risultati = []
        for t in stato["messages"][-1].tool_calls:
            print(f"  🔧 {t['name']}({t['args']})")
            if t["name"] not in self.tools:    # allucinazione: tool inventato -> istruiamo a riprovare
                esito = f"Il tool '{t['name']}' non esiste. Disponibili: {list(self.tools)}"
            else:
                esito = self.tools[t["name"]].invoke(t["args"])
            risultati.append(ToolMessage(tool_call_id=t["id"], name=t["name"], content=str(esito)))
        return {"messages": risultati}


print("Classe Agent definita.")

### I tool dell'agente

Tre tool che restituiscono **testo** (più leggibile per il modello). Il tool `valuta_acquisto` applica la policy **e tiene il conto degli importi già approvati nella sessione**: ci servirà per mostrare la memoria tra un turno e l'altro.

In [ ]:
_budget_impegnato: dict[str, float] = {}      # importi approvati nella "sessione" corrente, per badge


def _valuta(employee_id: str, prodotto_id: str):
    """Funzione di supporto (non è un tool): applica la policy tenendo conto del budget già impegnato."""
    emp = EMPLOYEES_DB.get(employee_id)
    if emp is None:
        return "RIFIUTO", f"dipendente '{employee_id}' non trovato", None, None
    prod = PRODUCTS_DB.get(prodotto_id)
    if prod is None:
        trovati = cerca_prodotti(prodotto_id)
        prod = next(iter(trovati.values()), None)
    if prod is None:
        return "RIFIUTO", f"prodotto '{prodotto_id}' non presente a catalogo", None, emp
    emp_netto = {**emp, "budget_residuo": emp["budget_residuo"] - _budget_impegnato.get(employee_id, 0)}
    esito, motivo = applica_policy(emp_netto, prod)
    return esito, motivo, prod, emp


@tool
def get_budget_dipendente(employee_id: str) -> str:
    """Restituisce budget residuo (al netto degli acquisti già approvati), ruolo, dipartimento e manager di un dipendente.

    Args:
        employee_id: Badge ID del dipendente (es. 'E4521')
    """
    emp = EMPLOYEES_DB.get(employee_id)
    if emp is None:
        return f"Dipendente '{employee_id}' non trovato nel sistema."
    residuo = emp["budget_residuo"] - _budget_impegnato.get(employee_id, 0)
    return (f"Dipendente: {emp['name']} ({emp['role']}, {emp['department']}). Budget residuo attuale: {residuo} EUR. "
            f"Manager: {emp['manager_email']}")


@tool
def cerca_prodotti_catalogo(query: str) -> str:
    """Cerca prodotti nel catalogo aziendale per nome o categoria e ne restituisce CODICE, nome, prezzo e categoria.

    Args:
        query: una o due parole chiave (es. 'thinkpad', 'monitor', 'dock')
    """
    trovati = cerca_prodotti(query)
    if not trovati:
        return f"Nessun prodotto trovato per '{query}': NON è presente a catalogo."
    return "\n".join(f"[{pid}] {p['name']} — {p['price_eur']} EUR ({p['category']})" for pid, p in trovati.items())


@tool
def verifica_policy(employee_id: str, prodotto_id: str) -> str:
    """Verifica (SOLA LETTURA) se un acquisto rispetta la policy. Restituisce APPROVATO, ESCALATION o RIFIUTO con motivazione.

    Args:
        employee_id: Badge ID del dipendente (es. 'E4521')
        prodotto_id: codice prodotto dal catalogo (es. 'LAP-DEV-01')
    """
    esito, motivo, prod, _ = _valuta(employee_id, prodotto_id)
    return f"{esito} — {motivo}"


@tool
def valuta_acquisto(employee_id: str, prodotto_id: str) -> str:
    """Valuta un acquisto e, se APPROVATO, ne registra l'importo come impegnato per le richieste successive.
    Restituisce APPROVATO, ESCALATION o RIFIUTO con motivazione.

    Args:
        employee_id: Badge ID del dipendente (es. 'E4521')
        prodotto_id: codice prodotto dal catalogo (es. 'LAP-DEV-01')
    """
    esito, motivo, prod, _ = _valuta(employee_id, prodotto_id)
    if esito == "APPROVATO":
        _budget_impegnato[employee_id] = _budget_impegnato.get(employee_id, 0) + prod["price_eur"]
        motivo += f". Importo impegnato in sessione: {_budget_impegnato[employee_id]} EUR"
    return f"{esito} — {motivo}"


@tool
def consulta_policy_hardware(categoria: str) -> str:
    """Restituisce la sezione della policy aziendale relativa a una categoria ('laptop', 'monitor', 'accessory').

    Args:
        categoria: categoria di prodotto
    """
    sezione, dentro = [], False
    for riga in POLICY_TEXT.splitlines():
        if riga.startswith("##"):
            if dentro:
                break
            dentro = categoria.lower() in riga.lower()
        if dentro:
            sezione.append(riga)
    return "\n".join(sezione) if sezione else POLICY_TEXT


tools_acquisto = [get_budget_dipendente, cerca_prodotti_catalogo, valuta_acquisto]

# ⚙️ test dei tool senza LLM (e verifica dell'impegno di budget)
_budget_impegnato.clear()
print(valuta_acquisto.invoke({"employee_id": "E4521", "prodotto_id": "LAP-DEV-01"}))
print(valuta_acquisto.invoke({"employee_id": "E4521", "prodotto_id": "ACC-DOCK-01"}))   # 150€ residui < 190€
_budget_impegnato.clear()

### 🧠 Eseguiamo l'agente

Lo stato è la lista dei messaggi: `HumanMessage` (utente) ➜ `AIMessage` con `tool_calls` ➜ `ToolMessage` (risultato) ➜ ... ➜ `AIMessage` finale.

In [ ]:
SYSTEM_PROMPT_AGENTE = """Sei l'Assistente Acquisti Hardware aziendale. Segui SEMPRE questo ordine:
1. Chiama get_budget_dipendente con il badge ID (es. 'E4521').
2. Chiama cerca_prodotti_catalogo con UNA o due parole chiave del prodotto, per ottenere il CODICE prodotto.
3. Chiama valuta_acquisto con badge ID e CODICE prodotto (NON passare il prezzo: viene letto in automatico).
4. Rispondi in italiano con la decisione (APPROVATO / ESCALATION / RIFIUTO) e la motivazione."""

agente = Agent(model=llm, tools=tools_acquisto, system=SYSTEM_PROMPT_AGENTE)
print("Agente creato. Tool:", list(agente.tools))
mostra_grafo(agente.graph)

In [ ]:
def chiedi(agente: Agent, testo: str, config: dict = None) -> list:
    """Invia un messaggio all'agente e stampa la risposta finale. Restituisce tutti i messaggi."""
    risultato = agente.graph.invoke({"messages": [HumanMessage(content=testo)]}, config)
    print("\n🤖", risultato["messages"][-1].content)
    return risultato["messages"]


if OLLAMA_OK:
    _budget_impegnato.clear()
    print("═" * 60, "\nMario Rossi — ThinkPad (atteso: APPROVATO)\n" + "═" * 60)
    messaggi = chiedi(agente, "Ciao, sono Mario Rossi (badge E4521). Posso acquistare un ThinkPad P16s per Project Delta?")

    _budget_impegnato.clear()
    print("\n" + "═" * 60, "\nGiulia Verdi — monitor 4K (atteso: ESCALATION)\n" + "═" * 60)
    chiedi(agente, "Salve, sono Giulia Verdi (E1082), Marketing. Posso acquistare il monitor 4K Dell UltraSharp?")

    print("\n" + "═" * 60, "\nDipendente inesistente (atteso: RIFIUTO)\n" + "═" * 60)
    chiedi(agente, "Ciao, sono Giovanni Bianchi (E9999). Voglio un laptop.")

    print("\n\nCronologia dei messaggi del primo scenario:")
    for m in messaggi:
        anteprima = str(m.content)[:90].replace("\n", " ")
        extra = f" tool_calls={[tc['name'] for tc in m.tool_calls]}" if getattr(m, "tool_calls", None) else ""
        print(f"  [{type(m).__name__}]{extra} {anteprima}")
else:
    print(AVVISO_OLLAMA)

### Grafo esplicito o classe `Agent`?

| | Classe `Agent` (ReAct) | Grafo esplicito (sezione precedente) |
| :--- | :--- | :--- |
| **Stato** | solo messaggi | `TypedDict` con campi strutturati |
| **Routing** | lo decide il LLM | funzioni Python esplicite |
| **Prevedibilità** | media | **alta** |
| **Flessibilità** | alta (si adatta a richieste impreviste) | bassa (flusso fisso) |
| **Debug** | cronologia messaggi | `.stream()` nodo per nodo |
| **Test senza LLM** | difficile | **facile** |
| **Quando usarlo** | problemi aperti, prototipi, tool generici | processi aziendali con esiti definiti |

**Regola pratica:** più il processo è definito, più logica sposta nel grafo; più il problema è aperto, più autonomia lasci al modello. I due approcci si combinano: un nodo di un grafo esplicito può essere un agente ReAct.

## Pattern Supervisor: più agenti specializzati

Il passo successivo è avere **più agenti**, ognuno con i propri tool, coordinati da un **Supervisor**:

```
START ─► supervisor ─► budget ─► catalogo ─► policy ─► aggregatore ─► END
             │            (ogni sotto-agente è un'istanza della classe Agent: un grafo llm ⇄ action)
             └─ decide QUALI agenti attivare (il "piano")
```

Due livelli di grafo:

- **grafo interno** di ogni sotto-agente: `llm ⇄ action`, stato = messaggi;
- **grafo principale**: stato **strutturato**, con un campo per il risultato di ciascun sotto-agente.

Ogni sotto-agente vede **solo i tool del proprio dominio**. Il router dopo ogni nodo guarda il *piano* del supervisor e sceglie il prossimo agente non ancora completato.

In [ ]:
agente_budget = Agent(
    model=llm, tools=[get_budget_dipendente],
    system=("Sei l'Agente Budget. Ricevi: 'Verifica il budget del dipendente <ID>'. "
            "Chiama get_budget_dipendente con l'ID (es. 'E4521') e rispondi con nome, ruolo, dipartimento e budget residuo."))

agente_catalogo = Agent(
    model=llm, tools=[cerca_prodotti_catalogo],
    system=("Sei l'Agente Catalogo. Estrai UNA parola chiave dal prodotto richiesto (es. 'ThinkPad' -> 'thinkpad', "
            "'monitor 4K' -> 'monitor') e chiama cerca_prodotti_catalogo. Rispondi con codice, nome, prezzo e categoria. "
            "Se non trovato, di' esplicitamente che il prodotto NON è a catalogo."))

agente_policy = Agent(
    model=llm, tools=[consulta_policy_hardware, verifica_policy],
    system=("Sei l'Agente Policy. Ricevi il badge del dipendente e il risultato della ricerca a catalogo. Passi: "
            "1) chiama consulta_policy_hardware con UNA parola tra 'laptop', 'monitor', 'accessory'; "
            "2) chiama verifica_policy con badge e CODICE prodotto (es. 'LAP-DEV-01'); "
            "3) rispondi con la regola applicabile e l'esito (APPROVATO / ESCALATION / RIFIUTO)."))

print("Sotto-agenti:", {"budget": list(agente_budget.tools), "catalogo": list(agente_catalogo.tools),
                        "policy": list(agente_policy.tools)})

In [ ]:
class StatoAcquisto(TypedDict, total=False):
    richiesta_utente: str
    employee_id: str
    piano: list[str]                              # agenti da attivare, nell'ordine deciso dal supervisor
    completati: Annotated[list[str], operator.add]
    risultato_budget: str
    risultato_catalogo: str
    risultato_policy: str
    messaggi_log: Annotated[list[str], operator.add]
    risposta_finale: str


ORDINE_CANONICO = ["budget", "catalogo", "policy"]   # la policy ha bisogno di budget e catalogo già pronti


def nodo_supervisor(stato: StatoAcquisto) -> dict:
    """Il LLM propone quali agenti servono; il codice valida la proposta e la mette in ordine."""
    print("\n🎯 SUPERVISOR")
    piano = list(ORDINE_CANONICO)         # piano di ripiego: tutti gli agenti
    if OLLAMA_OK:
        try:
            risposta = llm.invoke([HumanMessage(content=(
                f"Sei il supervisor di un sistema per acquisti hardware. Richiesta: \"{stato['richiesta_utente']}\"\n"
                f"Agenti disponibili: budget, catalogo, policy. Rispondi SOLO con una lista JSON, es. [\"budget\", \"catalogo\", \"policy\"]"))])
            trovato = re.search(r"\[.*?\]", risposta.content, re.DOTALL)
            proposta = json.loads(trovato.group()) if trovato else []
            valida = [a for a in ORDINE_CANONICO if a in proposta]     # solo nomi noti, in ordine canonico
            piano = valida or piano
        except Exception:
            pass
    print("   piano:", piano)
    return {"piano": piano, "messaggi_log": [f"[SUPERVISOR] piano={piano}"]}


def _esegui_sottoagente(agente: Agent, messaggio: str) -> str:
    risultato = agente.graph.invoke({"messages": [HumanMessage(content=messaggio)]})
    return risultato["messages"][-1].content


def nodo_budget(stato: StatoAcquisto) -> dict:
    print("\n💰 AGENTE BUDGET")
    testo = _esegui_sottoagente(agente_budget, f"Verifica il budget del dipendente {stato['employee_id']}")
    return {"risultato_budget": testo, "completati": ["budget"], "messaggi_log": [f"[BUDGET] {testo[:120]}"]}


def nodo_catalogo(stato: StatoAcquisto) -> dict:
    print("\n🛍️  AGENTE CATALOGO")
    testo = _esegui_sottoagente(agente_catalogo, f"Cerca il prodotto richiesto: {stato['richiesta_utente']}")
    return {"risultato_catalogo": testo, "completati": ["catalogo"], "messaggi_log": [f"[CATALOGO] {testo[:120]}"]}


def nodo_policy(stato: StatoAcquisto) -> dict:
    print("\n📋 AGENTE POLICY")
    testo = _esegui_sottoagente(agente_policy, (
        f"Dipendente {stato['employee_id']}. Risultato catalogo: {stato.get('risultato_catalogo', 'n/d')[:200]}. "
        f"Richiesta originale: {stato['richiesta_utente']}"))
    return {"risultato_policy": testo, "completati": ["policy"], "messaggi_log": [f"[POLICY] {testo[:120]}"]}


def nodo_aggregatore(stato: StatoAcquisto) -> dict:
    print("\n📊 AGGREGATORE")
    ripiego = "\n".join(f"{k.upper()}: {stato.get(f'risultato_{k}', 'non eseguito')}" for k in ORDINE_CANONICO)
    testo = genera_testo(
        f"Sei il sistema di sintesi per acquisti hardware. Richiesta: {stato['richiesta_utente']}\n"
        f"BUDGET: {stato.get('risultato_budget', 'n/d')}\nCATALOGO: {stato.get('risultato_catalogo', 'n/d')}\n"
        f"POLICY: {stato.get('risultato_policy', 'n/d')}\n"
        "Scrivi in italiano la decisione finale (APPROVATO / ESCALATION / RIFIUTATO) con motivazione breve.", ripiego)
    return {"risposta_finale": testo, "messaggi_log": ["[AGGREGATORE] sintesi completata"]}


def prossimo_agente(stato: StatoAcquisto) -> str:
    """Router: primo agente del piano non ancora completato, altrimenti l'aggregatore."""
    for agente in stato.get("piano", []):
        if agente not in stato.get("completati", []):
            return agente
    return "aggregatore"


print("Nodi del supervisor definiti.")

In [ ]:
def costruisci_supervisor():
    g = StateGraph(StatoAcquisto)
    g.add_node("supervisor", nodo_supervisor)
    g.add_node("budget", nodo_budget)
    g.add_node("catalogo", nodo_catalogo)
    g.add_node("policy", nodo_policy)
    g.add_node("aggregatore", nodo_aggregatore)

    mappa = {"budget": "budget", "catalogo": "catalogo", "policy": "policy", "aggregatore": "aggregatore"}
    g.add_edge(START, "supervisor")
    for nodo in ["supervisor", "budget", "catalogo", "policy"]:       # dopo ognuno, il router sceglie il prossimo
        g.add_conditional_edges(nodo, prossimo_agente, mappa)
    g.add_edge("aggregatore", END)
    return g.compile()


app_supervisor = costruisci_supervisor()
mostra_grafo(app_supervisor)

In [ ]:
def esegui_supervisor(richiesta: str, employee_id: str) -> dict:
    print("\n" + "═" * 62 + f"\n  📥 {richiesta}\n" + "═" * 62)
    risultato = app_supervisor.invoke({"richiesta_utente": richiesta, "employee_id": employee_id,
                                       "completati": [], "messaggi_log": []})
    print("\n" + "─" * 62 + "\n🤖 RISPOSTA FINALE:\n" + risultato["risposta_finale"])
    return risultato


if OLLAMA_OK:
    _budget_impegnato.clear()
    r_sup = esegui_supervisor("Ciao, sono Mario Rossi (E4521). Vorrei acquistare un ThinkPad P16s per Project Delta.", "E4521")
    print("\nLog accumulato:", *r_sup["messaggi_log"], sep="\n  ")
    esegui_supervisor("Salve, sono Giulia Verdi (E1082). Ho bisogno del monitor 4K Dell UltraSharp.", "E1082")
else:
    print(AVVISO_OLLAMA)

### Esercizio — agenti in parallelo

**Consegna:** `budget` e `catalogo` sono indipendenti: possono girare **in parallelo**, poi convergere su `policy`.

In LangGraph il parallelismo si ottiene con **due archi uscenti dallo stesso nodo** (*fan-out*); per aspettare entrambi prima di proseguire si usa un arco con **lista di sorgenti** (*fan-in*). I campi scritti in parallelo da nodi diversi vanno bene, e quelli accumulati (`completati`, `messaggi_log`) sono protetti dai reducer.

In [ ]:
# SOLUZIONE
def costruisci_supervisor_parallelo():
    g = StateGraph(StatoAcquisto)
    for nome, funzione in [("supervisor", nodo_supervisor), ("budget", nodo_budget), ("catalogo", nodo_catalogo),
                           ("policy", nodo_policy), ("aggregatore", nodo_aggregatore)]:
        g.add_node(nome, funzione)

    g.add_edge(START, "supervisor")
    g.add_edge("supervisor", "budget")                 # fan-out: due archi dallo stesso nodo...
    g.add_edge("supervisor", "catalogo")               # ...=> budget e catalogo partono insieme
    g.add_edge(["budget", "catalogo"], "policy")       # fan-in: policy parte quando ENTRAMBI hanno finito
    g.add_edge("policy", "aggregatore")
    g.add_edge("aggregatore", END)
    return g.compile()


app_parallelo = costruisci_supervisor_parallelo()
mostra_grafo(app_parallelo)

## Persistenza: la memoria tra una chiamata e l'altra

Senza checkpointer ogni `invoke` parte da zero. Con un **checkpointer** LangGraph **salva lo stato dopo ogni passo** e lo ritrova alla chiamata successiva.

| Concetto | Significato |
| :--- | :--- |
| **Checkpointer** | il "salvataggio": `MemorySaver()` (in memoria, si perde al riavvio del kernel). In produzione: SQLite, Postgres... |
| **`thread_id`** | l'identificatore della conversazione, passato in `config`. Stesso `thread_id` = stessa memoria; `thread_id` diverso = conversazione indipendente |
| **Checkpoint** | una "fotografia" dello stato in un preciso momento |

```python
app = grafo.compile(checkpointer=MemorySaver())
config = {"configurable": {"thread_id": "conversazione-1"}}
app.invoke(input, config)
```

Prima il meccanismo puro, con un grafo giocattolo senza LLM.

In [ ]:
class StatoChat(TypedDict):
    messages: Annotated[list[str], operator.add]
    turni: int


def rispondi(stato: StatoChat) -> dict:
    turno = stato.get("turni", 0) + 1
    return {"messages": [f"risposta #{turno}"], "turni": turno}


g = StateGraph(StatoChat)
g.add_node("rispondi", rispondi)
g.add_edge(START, "rispondi")
g.add_edge("rispondi", END)
app_chat = g.compile(checkpointer=MemorySaver())

thread_a = {"configurable": {"thread_id": "A"}}
thread_b = {"configurable": {"thread_id": "B"}}

app_chat.invoke({"messages": ["ciao"]}, thread_a)
app_chat.invoke({"messages": ["come va?"]}, thread_a)      # stesso thread: ricorda il turno precedente
app_chat.invoke({"messages": ["ciao"]}, thread_b)          # thread diverso: parte da zero

print("Thread A:", app_chat.get_state(thread_a).values)
print("Thread B:", app_chat.get_state(thread_b).values)
assert app_chat.get_state(thread_a).values["turni"] == 2
assert app_chat.get_state(thread_b).values["turni"] == 1

print("\nStoria dei checkpoint del thread A (dal più recente):")
for istantanea in app_chat.get_state_history(thread_a):
    print(f"  passo={istantanea.metadata.get('step'):>2}  prossimo={str(istantanea.next):<15} turni={istantanea.values.get('turni')}")

### 🧠 Multi-turno con l'agente

Stesso agente, ora con `MemorySaver`. Mario chiede il ThinkPad (1650€, approvato) e **poi**, nello stesso thread, il Dock (190€). Il budget residuo di sessione è 1800 − 1650 = **150€**, quindi il Dock va in **ESCALATION**. Un thread diverso parte invece da zero.

>NB) Con modelli piccoli come llama3.2 conviene scrivere **sempre badge e codice prodotto** nei messaggi: i riferimenti impliciti ("e se volessi anche il dock?") in thread lunghi possono confonderli.

In [ ]:
agente_memoria = Agent(model=llm, tools=tools_acquisto, system=SYSTEM_PROMPT_AGENTE, checkpointer=MemorySaver())

if OLLAMA_OK:
    _budget_impegnato.clear()
    thread = {"configurable": {"thread_id": "acquisto-001"}}

    print("── Turno 1 ──")
    chiedi(agente_memoria, "Ciao, sono Mario Rossi (E4521). Posso acquistare il ThinkPad P16s?", thread)
    print("\n── Turno 2 (stesso thread) ──")
    chiedi(agente_memoria, "Sempre Mario Rossi (E4521): vorrei anche il Thunderbolt 4 Dock (ACC-DOCK-01).", thread)

    istantanea = agente_memoria.graph.get_state(thread)
    print(f"\nMessaggi salvati nel thread: {len(istantanea.values['messages'])}   prossimo nodo: {istantanea.next}")
    _budget_impegnato.clear()
else:
    print(AVVISO_OLLAMA)

## Human-in-the-Loop con LangGraph

Nella Parte 3 il checkpoint umano era "ad hoc". In LangGraph la pausa è una funzione **nativa** del grafo: l'esecuzione si sospende, lo stato resta salvato dal checkpointer, e riprende quando arriva la risposta umana, anche dopo ore.

Ci sono due meccanismi:

| | `interrupt_before` / `interrupt_after` | `interrupt()` |
| :--- | :--- | :--- |
| Tipo | **statico**: fissato in `compile()` | **dinamico**: chiamato dentro un nodo |
| Dove si ferma | prima/dopo un nodo | nel punto esatto in cui lo chiami |
| Cosa può mostrare | lo stato corrente (es. le tool call pianificate) | un **payload** a scelta |
| Come si riprende | `graph.stream(None, config)` | `graph.invoke(Command(resume=valore), config)` |
| Caso tipico | "ispeziona o modifica cosa sta per fare l'agente" | "chiedi al manager sì/no e usa la sua risposta" |

Entrambi richiedono **checkpointer** e **`thread_id`**.

### Un modello finto per esercitarsi

Per vedere la pausa in modo **deterministico e senza Ollama**, usiamo un `ModelloSimulato`: un finto LLM che chiede il tool `valuta_acquisto` e poi riassume il risultato. Ha la stessa interfaccia del modello vero (`bind_tools`, `invoke`), quindi la classe `Agent` non nota la differenza. Per la versione reale basta sostituire `ModelloSimulato()` con `llm`.

>NB) Simulare il modello è anche una tecnica professionale: permette di **testare un grafo in modo ripetibile**, senza costi né variabilità.

In [ ]:
class ModelloSimulato:
    """Finto LLM deterministico: al primo giro chiede 'valuta_acquisto', poi riassume il risultato."""

    def bind_tools(self, tools):
        return self

    def invoke(self, messaggi):
        ultimo = messaggi[-1]
        if isinstance(ultimo, ToolMessage):                       # tool già eseguito -> risposta finale
            return AIMessage(content=f"Esito della valutazione: {ultimo.content}")
        badge = re.search(r"\bE\d{3,5}\b", ultimo.content)
        trovati = cerca_prodotti(ultimo.content)
        return AIMessage(content="", tool_calls=[{
            "name": "valuta_acquisto", "type": "tool_call", "id": uuid.uuid4().hex,
            "args": {"employee_id": badge.group(0) if badge else "UNKNOWN",
                     "prodotto_id": next(iter(trovati), "SCONOSCIUTO")}}])


agente_hitl = Agent(model=ModelloSimulato(), tools=tools_acquisto, system="(il modello simulato ignora il system prompt)",
                    checkpointer=MemorySaver(),
                    interrupt_before=["action"])       # <- il grafo si ferma PRIMA di eseguire i tool
print("Agente con interrupt_before=['action'] pronto.")


def stampa_eventi(eventi):
    """Stampa in modo leggibile gli eventi di .stream()."""
    for evento in eventi:
        for nodo, aggiornamento in evento.items():
            if nodo == "__interrupt__":
                print("  ⏸️  PAUSA (interrupt)")
                continue
            ultimo = aggiornamento["messages"][-1]
            if getattr(ultimo, "tool_calls", None):
                print(f"  [{nodo}] il modello vuole chiamare: {[(t['name'], t['args']) for t in ultimo.tool_calls]}")
            elif isinstance(ultimo, ToolMessage):
                print(f"  [{nodo}] risultato del tool: {ultimo.content}")
            else:
                print(f"  [{nodo}] {ultimo.content}")

### Passo 1 — Prima esecuzione: il grafo si ferma

Dopo il nodo `llm` il grafo si sospende: il modello ha **pianificato** una tool call ma il tool **non è stato eseguito**. Possiamo ispezionare lo stato: `next` ci dice quale nodo è in attesa.

In [ ]:
_budget_impegnato.clear()
thread_h = {"configurable": {"thread_id": "approval-001"}}

print("=== Prima esecuzione ===")
stampa_eventi(agente_hitl.graph.stream(
    {"messages": [HumanMessage(content="Giulia Verdi (E1082) vuole il monitor Dell UltraSharp 4K.")]}, thread_h))

stato_pausa = agente_hitl.graph.get_state(thread_h)
print("\nNodo in attesa (next):", stato_pausa.next)
print("Tool call pianificata :", stato_pausa.values["messages"][-1].tool_calls[0]["name"],
      stato_pausa.values["messages"][-1].tool_calls[0]["args"])
assert stato_pausa.next == ("action",)

### Passo 2 — Il manager approva: si riprende

Per riprendere si richiama `stream(None, config)`: `None` come input significa *"continua da dove eri"*.

In [ ]:
print("=== Il manager approva: ripresa ===")
stampa_eventi(agente_hitl.graph.stream(None, thread_h))
print("\nNodo in attesa dopo la ripresa:", agente_hitl.graph.get_state(thread_h).next, "(vuoto = grafo concluso)")

### Passo 3 — Il manager **modifica** la tool call con `update_state()`

Durante la pausa possiamo **riscrivere** ciò che l'agente sta per fare. Qui il manager cambia il prodotto: invece del monitor, un laptop standard.

Per sostituire un messaggio basta ricrearlo con **lo stesso `id`**: il reducer `add_messages` lo riconosce e lo rimpiazza invece di accodarlo.

In [ ]:
thread_m = {"configurable": {"thread_id": "approval-002"}}
stampa_eventi(agente_hitl.graph.stream(
    {"messages": [HumanMessage(content="Giulia Verdi (E1082) chiede il monitor Dell UltraSharp 4K.")]}, thread_m))

in_pausa = agente_hitl.graph.get_state(thread_m)
messaggio_originale = in_pausa.values["messages"][-1]
print("\nPianificato dal modello :", messaggio_originale.tool_calls[0]["args"])

# Stesso id del messaggio originale => viene SOSTITUITO
messaggio_modificato = AIMessage(
    content=messaggio_originale.content, id=messaggio_originale.id,
    tool_calls=[{"name": "valuta_acquisto", "type": "tool_call", "id": messaggio_originale.tool_calls[0]["id"],
                 "args": {"employee_id": "E1082", "prodotto_id": "LAP-STD-01"}}])    # il manager sceglie il laptop standard
agente_hitl.graph.update_state(thread_m, {"messages": [messaggio_modificato]})

print("Dopo update_state       :", agente_hitl.graph.get_state(thread_m).values["messages"][-1].tool_calls[0]["args"])
print("Numero messaggi (nessun duplicato):", len(agente_hitl.graph.get_state(thread_m).values["messages"]))

print("\n=== Ripresa con la tool call modificata ===")
stampa_eventi(agente_hitl.graph.stream(None, thread_m))

### Passo 4 — Viaggio nel tempo: `get_state_history()` e fork

Ogni checkpoint è una fotografia navigabile. Possiamo **tornare a un punto precedente**, modificare lo stato e far ripartire l'esecuzione da lì, creando un **ramo alternativo** (*fork*). I checkpoint precedenti restano nella storia; a cambiare è la "testa" del thread, che ora punta al ramo.

In [ ]:
storia = list(agente_hitl.graph.get_state_history(thread_h))       # dal più recente al più vecchio
print("Checkpoint del thread 'approval-001' (più recente prima):")
for s in storia:
    print(f"  passo={s.metadata.get('step'):>2}  prossimo={str(s.next):<14} messaggi={len(s.values.get('messages', []))}")

# Cerchiamo lo stato appena dopo l'arrivo del messaggio utente: tocca a 'llm' e c'è UN solo messaggio
punto_di_partenza = next(s for s in storia if s.next == ("llm",) and len(s.values["messages"]) == 1)
messaggio_utente = punto_di_partenza.values["messages"][0]

# Fork: stessa posizione, ma il messaggio utente (stesso id) ora chiede un'altra cosa
config_fork = agente_hitl.graph.update_state(
    punto_di_partenza.config,
    {"messages": [HumanMessage(content="Mario Rossi (E4521) vuole il ThinkPad P16s.", id=messaggio_utente.id)]})

print("\n=== Esecuzione del ramo alternativo (si ferma di nuovo prima dei tool) ===")
stampa_eventi(agente_hitl.graph.stream(None, config_fork))
# Per CONTINUARE il ramo si usa il config del thread: la sua "testa" ora è il ramo.
# (config_fork riparte sempre dal checkpoint di fork, rieseguendo lo stesso passo: è un 'replay'.)
print("\n=== Il manager approva anche il ramo ===")
stampa_eventi(agente_hitl.graph.stream(None, thread_h))
print(f"\nLa storia originale è ancora nei checkpoint ({len(list(agente_hitl.graph.get_state_history(thread_h)))} in totale); "
      f"la testa del thread è ora il ramo.")

### `interrupt()` + `Command(resume=...)`: la pausa dentro un nodo

`interrupt_before` ferma il grafo **tra** i nodi. Per un'approvazione vera ("il manager risponde sì/no con una nota") è più naturale fermarsi **dentro** un nodo e ricevere un **valore di ritorno**.

```python
def nodo_approvazione(stato):
    decisione = interrupt({...payload da mostrare al manager...})   # <- il grafo si ferma qui
    # alla ripresa, 'decisione' contiene il valore passato con Command(resume=...)
```

>NB) Alla ripresa il nodo **riparte dall'inizio**, non dalla riga dell'`interrupt`. Quindi tutto ciò che sta *prima* dell'`interrupt()` nello stesso nodo viene rieseguito: niente effetti collaterali (invio mail, scritture su DB) prima di un `interrupt()`, oppure rendili idempotenti.

Costruiamo un mini-flusso deterministico: sotto i 300€ approvazione automatica, sopra serve il manager.

In [ ]:
class StatoApprovazione(TypedDict, total=False):
    richiesta: str
    importo: float
    esito: str
    nota: str


def nodo_valuta(stato: StatoApprovazione) -> dict:
    return {"esito": "DA_APPROVARE" if stato["importo"] > 300 else "APPROVATO_AUTO"}


def nodo_approvazione(stato: StatoApprovazione) -> dict:
    decisione = interrupt({"domanda": "Approvi questa spesa?", "richiesta": stato["richiesta"], "importo": stato["importo"]})
    # Alla ripresa, interrupt() restituisce il valore di Command(resume=...)
    return {"esito": "APPROVATO_MANAGER" if decisione["approvato"] else "RIFIUTATO_MANAGER",
            "nota": decisione.get("nota", "")}


g = StateGraph(StatoApprovazione)
g.add_node("valuta", nodo_valuta)
g.add_node("approvazione", nodo_approvazione)
g.add_edge(START, "valuta")
g.add_conditional_edges("valuta", lambda s: "approvazione" if s["esito"] == "DA_APPROVARE" else END,
                        {"approvazione": "approvazione", END: END})
g.add_edge("approvazione", END)
app_approvazione = g.compile(checkpointer=MemorySaver())      # l'interrupt richiede un checkpointer


def payload_in_attesa(app, config):
    """Restituisce il payload dell'interrupt in attesa (o None)."""
    for task in app.get_state(config).tasks:
        for pausa in task.interrupts:
            return pausa.value
    return None


# Caso 1: sotto soglia -> nessuna pausa
c1 = {"configurable": {"thread_id": "spesa-1"}}
print("Spesa 120€ :", app_approvazione.invoke({"richiesta": "Mouse ergonomico", "importo": 120}, c1)["esito"])

# Caso 2: sopra soglia -> il grafo si ferma e mostra il payload
c2 = {"configurable": {"thread_id": "spesa-2"}}
app_approvazione.invoke({"richiesta": "Monitor 4K", "importo": 520}, c2)
print("\nSpesa 520€ : grafo in pausa. Nodo in attesa:", app_approvazione.get_state(c2).next)
print("Payload mostrato al manager:", payload_in_attesa(app_approvazione, c2))

# Il manager risponde: si riprende con Command(resume=...)
finale = app_approvazione.invoke(Command(resume={"approvato": True, "nota": "OK per la campagna Q2"}), c2)
print("Dopo la decisione:", finale["esito"], "|", finale["nota"])
assert finale["esito"] == "APPROVATO_MANAGER"

# Caso 3: il manager rifiuta
c3 = {"configurable": {"thread_id": "spesa-3"}}
app_approvazione.invoke({"richiesta": "Monitor 4K", "importo": 520}, c3)
rifiuto = app_approvazione.invoke(Command(resume={"approvato": False, "nota": "Budget finito"}), c3)
print("\nRifiuto del manager:", rifiuto["esito"], "|", rifiuto["nota"])
assert rifiuto["esito"] == "RIFIUTATO_MANAGER"

### Esercizio — flusso interattivo con input reale

**Consegna:** scrivi `gestisci_richiesta(app, stato_iniziale, thread_id)` che avvia il grafo; se trova una pausa stampa il payload, chiede `input("Autorizzare? [s/n]: ")` e riprende con `Command(resume=...)`. È lo stesso schema della cella interattiva della Parte 3, ma ora la pausa è vera.

In [ ]:
# SOLUZIONE (usa auto=True/False per decidere se simulare l'operatore o chiedere con input())
def gestisci_richiesta(app, stato_iniziale: dict, thread_id: str, auto=None) -> dict:
    config = {"configurable": {"thread_id": thread_id}}
    risultato = app.invoke(stato_iniziale, config)

    payload = payload_in_attesa(app, config)
    if payload is not None:
        print("⏸️  In attesa dell'operatore:", payload)
        if auto is None:
            approvato = input("Autorizzare? [s/n]: ").strip().lower() in ("s", "si", "y", "yes")
            nota = input("Nota: ").strip()
        else:
            approvato, nota = auto, "decisione simulata"
        risultato = app.invoke(Command(resume={"approvato": approvato, "nota": nota}), config)
    return risultato


print(gestisci_richiesta(app_approvazione, {"richiesta": "Dock", "importo": 450}, "spesa-4", auto=True)["esito"])
print(gestisci_richiesta(app_approvazione, {"richiesta": "Cavo", "importo": 15}, "spesa-5")["esito"])
# Per provarlo davvero:  gestisci_richiesta(app_approvazione, {"richiesta": "Dock", "importo": 450}, "spesa-6")

## Riepilogo Parte 4

| Concetto | In una riga |
| :--- | :--- |
| **State** | `TypedDict` condiviso; ogni nodo scrive i **propri** campi |
| **Reducer** | decide come si fonde l'output di un nodo: sovrascrive, `operator.add` accoda, `add_messages` accoda/sostituisce per `id` |
| **Nodo** | funzione `stato -> dict parziale` |
| **Arco fisso / condizionale** | `add_edge` / `add_conditional_edges(nodo, router, mappa)` |
| **Ciclo** | un arco che torna indietro, sempre con una **condizione di uscita** (+ `recursion_limit`) |
| **ReAct come grafo** | due nodi `llm ⇄ action` con un arco condizionale |
| **Supervisor** | un nodo che pianifica + router che attiva i sotto-agenti; fan-out/fan-in per il parallelismo |
| **Checkpointer + `thread_id`** | memoria tra chiamate, `get_state`, `get_state_history` |
| **`interrupt_before`** | pausa **tra** i nodi; si riprende con `stream(None, config)` e si modifica con `update_state` |
| **`interrupt()` + `Command(resume=)`** | pausa **dentro** un nodo con payload e valore di ritorno |
| **Time travel** | tornare a un checkpoint, modificarlo e creare un ramo (fork) |

# Parte 5 — Esercitazione finale: orchestrazione completa

Mettiamo insieme tutto in **un solo grafo** che raggiunge i quattro obiettivi dell'esercitazione:

| Obiettivo | Come lo soddisfiamo |
| :--- | :--- |
| **1.** Un secondo framework | il sistema è realizzato con **LangGraph** |
| **2.** Stato e routing condizionale | `StatoFinale` strutturato + tre router |
| **3.** Un workflow non banale | validazione, budget, catalogo, policy, approvazione umana, sintesi |
| **4.** Almeno un failure path | input non valido, **retry** del catalogo con esaurimento dei tentativi, prodotto non a catalogo, rifiuto del manager |

```
START ─► valida_input ──(errore)──────────────────────────────► gestisci_errore ─► END
              │                                                        ▲
              ▼ (ok)                                                   │ (esauriti i tentativi)
           budget ─► catalogo ──(fallito, retry < 3)──► catalogo ──────┤
                        │                                              │
                        ▼ (ok)                                         │
                      policy ──(ESCALATION)──► approvazione_manager ─┐  │
                        │                        (interrupt)         │  │
                        └──(APPROVATO / RIFIUTO)──────────────► aggregatore ─► END
```

Il fallimento del catalogo è **simulato in modo deterministico** con un campo dello stato (`simula_fallimenti_catalogo`), così i test sono ripetibili. Il LLM, se c'è, scrive solo il messaggio finale.

### Lo stato

>NB) Il nodo `approvazione_manager` e il campo che ne contiene l'esito hanno **nomi diversi** (`decisione_manager`): un nome di nodo uguale a un campo dello stato è un errore.

In [ ]:
class StatoFinale(TypedDict, total=False):
    # Input
    richiesta_utente: str
    employee_id: str
    prodotto_query: str
    errore_input: str                       # vuoto se l'input è valido
    # Risultati intermedi
    budget_data: dict
    catalog_data: dict
    prodotto_scelto: dict
    # Gestione del fallimento del catalogo (retry)
    simula_fallimenti_catalogo: int         # quante volte il catalogo deve fallire (solo per i test)
    retry_catalogo: int
    catalogo_fallito: bool
    # Decisione
    esito: str                              # APPROVATO | ESCALATION | RIFIUTO | APPROVATO_MANAGER | RIFIUTATO_MANAGER | ERRORE
    motivazione: str
    decisione_manager: dict
    risposta_finale: str
    log: Annotated[list[str], operator.add]


MAX_TENTATIVI_CATALOGO = 3
print("Campi dello stato:", len(StatoFinale.__annotations__))

### I nodi

Osserva tre scelte di design:

- `valida_input` è il **guardiano**: blocca subito input vuoti o badge inesistenti, prima di sprecare passi;
- `nodo_catalogo` ritorna `catalogo_fallito=True` e incrementa `retry_catalogo`: sarà il **router** a decidere se riprovare;
- `approvazione_manager` contiene **solo** l'`interrupt()`, senza effetti collaterali prima (si riesegue dall'inizio alla ripresa).

In [ ]:
def valida_input(stato: StatoFinale) -> dict:
    """Guardiano: verifica la richiesta, ricava badge e prodotto, controlla che il dipendente esista."""
    testo = stato.get("richiesta_utente", "").strip()
    if len(testo) < 10:
        return {"errore_input": "richiesta vuota o troppo breve", "log": ["valida_input: richiesta non valida"]}
    badge = re.search(r"\bE\d{3,5}\b", testo)
    employee_id = badge.group(0) if badge else "UNKNOWN"
    if employee_id not in EMPLOYEES_DB:
        return {"errore_input": f"dipendente '{employee_id}' non trovato", "employee_id": employee_id,
                "log": [f"valida_input: badge {employee_id} sconosciuto"]}
    # Parole del testo presenti nel catalogo; se non ce n'è nessuna usiamo tutto il testo (il catalogo dirà NOT_FOUND)
    query = " ".join(t for t in tokenizza(testo) if cerca_prodotti(t)) or " ".join(tokenizza(testo))
    return {"errore_input": "", "employee_id": employee_id, "prodotto_query": query,
            "log": [f"valida_input: ok ({employee_id}, query={query!r})"]}


def router_validazione(stato: StatoFinale) -> str:
    return "gestisci_errore" if stato.get("errore_input") else "budget"


def gestisci_errore(stato: StatoFinale) -> dict:
    """Failure path unico: input non valido oppure servizio catalogo non disponibile."""
    if stato.get("errore_input"):
        motivo = stato["errore_input"]
    else:
        motivo = f"servizio catalogo non disponibile dopo {stato.get('retry_catalogo', 0)} tentativi"
    return {"esito": "ERRORE", "motivazione": motivo, "risposta_finale": f"❌ Operazione non completata: {motivo}.",
            "log": [f"gestisci_errore: {motivo}"]}


def nodo_budget(stato: StatoFinale) -> dict:
    emp = EMPLOYEES_DB[stato["employee_id"]]
    return {"budget_data": {"status": "OK", **emp}, "log": [f"budget: {emp['name']}, residuo {emp['budget_residuo']}€"]}


def nodo_catalogo(stato: StatoFinale) -> dict:
    tentativo = stato.get("retry_catalogo", 0)
    if tentativo < stato.get("simula_fallimenti_catalogo", 0):          # fallimento simulato
        return {"retry_catalogo": tentativo + 1, "catalogo_fallito": True,
                "log": [f"catalogo: tentativo {tentativo + 1} FALLITO"]}
    trovati = cerca_prodotti(stato["prodotto_query"])
    if not trovati:
        return {"catalogo_fallito": False, "catalog_data": {"status": "NOT_FOUND"},
                "log": ["catalogo: prodotto non trovato"]}
    return {"catalogo_fallito": False, "catalog_data": {"status": "OK", "risultati": trovati},
            "log": [f"catalogo: ok al tentativo {tentativo + 1}"]}


def router_catalogo(stato: StatoFinale) -> str:
    if stato.get("catalogo_fallito"):
        return "catalogo" if stato["retry_catalogo"] < MAX_TENTATIVI_CATALOGO else "gestisci_errore"
    return "policy"


def nodo_policy(stato: StatoFinale) -> dict:
    catalogo = stato["catalog_data"]
    if catalogo["status"] == "NOT_FOUND":
        return {"esito": "RIFIUTO", "motivazione": "prodotto non presente a catalogo", "log": ["policy: RIFIUTO"]}
    prodotto = next(iter(catalogo["risultati"].values()))
    esito, motivo = applica_policy(stato["budget_data"], prodotto)
    return {"esito": esito, "motivazione": motivo, "prodotto_scelto": prodotto, "log": [f"policy: {esito}"]}


def router_policy(stato: StatoFinale) -> str:
    return "approvazione_manager" if stato["esito"] == "ESCALATION" else "aggregatore"


def approvazione_manager(stato: StatoFinale) -> dict:
    """Pausa vera: il grafo si ferma finché il manager non risponde. NESSUN effetto collaterale prima dell'interrupt."""
    decisione = interrupt({
        "domanda": "Approvi l'acquisto?",
        "dipendente": stato["budget_data"]["name"],
        "prodotto": stato["prodotto_scelto"]["name"],
        "prezzo_eur": stato["prodotto_scelto"]["price_eur"],
        "motivazione": stato["motivazione"],
        "manager": stato["budget_data"]["manager_email"],
    })
    return {"decisione_manager": decisione,
            "esito": "APPROVATO_MANAGER" if decisione.get("approvato") else "RIFIUTATO_MANAGER",
            "log": [f"approvazione_manager: {'approvato' if decisione.get('approvato') else 'rifiutato'}"]}


def aggregatore(stato: StatoFinale) -> dict:
    nome = stato.get("prodotto_scelto", {}).get("name", "il prodotto")
    nota = stato.get("decisione_manager", {}).get("nota", "")
    ripiego = {"APPROVATO": f"✅ APPROVATO: {nome} ({stato['motivazione']}).",
               "APPROVATO_MANAGER": f"✅ APPROVATO dal manager: {nome}. {nota}",
               "RIFIUTATO_MANAGER": f"❌ RIFIUTATO dal manager: {nome}. {nota}",
               "RIFIUTO": f"❌ RIFIUTO: {stato['motivazione']}."}[stato["esito"]]
    testo = genera_testo(f"Scrivi in italiano, in massimo due frasi, la risposta per un dipendente. "
                         f"Esito: {stato['esito']}. Prodotto: {nome}. Motivazione: {stato['motivazione']}. {nota} "
                         f"Non cambiare l'esito.", ripiego)
    return {"risposta_finale": testo, "log": [f"aggregatore: {stato['esito']}"]}


g = StateGraph(StatoFinale)
for nome, funzione in [("valida_input", valida_input), ("gestisci_errore", gestisci_errore), ("budget", nodo_budget),
                       ("catalogo", nodo_catalogo), ("policy", nodo_policy),
                       ("approvazione_manager", approvazione_manager), ("aggregatore", aggregatore)]:
    g.add_node(nome, funzione)

g.add_edge(START, "valida_input")
g.add_conditional_edges("valida_input", router_validazione, {"gestisci_errore": "gestisci_errore", "budget": "budget"})
g.add_edge("budget", "catalogo")
g.add_conditional_edges("catalogo", router_catalogo,
                        {"catalogo": "catalogo", "gestisci_errore": "gestisci_errore", "policy": "policy"})   # ciclo di retry
g.add_conditional_edges("policy", router_policy,
                        {"approvazione_manager": "approvazione_manager", "aggregatore": "aggregatore"})
g.add_edge("approvazione_manager", "aggregatore")
g.add_edge("aggregatore", END)
g.add_edge("gestisci_errore", END)

app_finale = g.compile(checkpointer=MemorySaver())       # il checkpointer serve per l'interrupt
mostra_grafo(app_finale)

### Test: tutti i percorsi, tutti verificati

La funzione `esegui_finale` avvia il grafo e, se trova una pausa, mostra la richiesta al manager e riprende con la decisione data (`decisione_manager`). Le `assert` coprono **ogni percorso** del grafo, **senza Ollama**.

In [ ]:
def esegui_finale(richiesta: str, thread_id: str, fallimenti: int = 0, decisione_manager: dict = None) -> dict:
    config = {"configurable": {"thread_id": thread_id}}
    print("\n" + "═" * 64 + f"\n  🚀 [{thread_id}] {richiesta}" + (f"  (catalogo fallisce {fallimenti} volte)" if fallimenti else "")
          + "\n" + "═" * 64)
    risultato = app_finale.invoke({"richiesta_utente": richiesta, "simula_fallimenti_catalogo": fallimenti,
                                   "retry_catalogo": 0, "log": []}, config)

    payload = payload_in_attesa(app_finale, config)
    if payload is not None:
        print("  ⏸️  PAUSA — richiesta al manager:", json.dumps(payload, ensure_ascii=False))
        assert app_finale.get_state(config).next == ("approvazione_manager",)
        if decisione_manager is not None:
            risultato = app_finale.invoke(Command(resume=decisione_manager), config)

    for riga in risultato["log"]:
        print("   •", riga)
    print(f"  ESITO: {risultato['esito']}\n  {risultato['risposta_finale']}")
    return risultato


MARIO = "Ciao, sono Mario Rossi (E4521). Voglio acquistare il ThinkPad P16s."
GIULIA = "Salve, sono Giulia Verdi (E1082). Ho bisogno del monitor 4K Dell UltraSharp."

r1 = esegui_finale(MARIO, "t1");                                              assert r1["esito"] == "APPROVATO"
r2 = esegui_finale("Sono Giovanni Bianchi (E9999), voglio un laptop.", "t2"); assert r2["esito"] == "ERRORE"
r3 = esegui_finale(MARIO, "t3", fallimenti=2);                                assert r3["esito"] == "APPROVATO" and r3["retry_catalogo"] == 2
r4 = esegui_finale(MARIO, "t4", fallimenti=5);                                assert r4["esito"] == "ERRORE" and r4["retry_catalogo"] == 3
r5 = esegui_finale("Sono Alessandro Corti (E9900), mi serve un Apple Vision Pro.", "t5"); assert r5["esito"] == "RIFIUTO"
r6 = esegui_finale(GIULIA, "t6", decisione_manager={"approvato": True, "nota": "OK per la Brand Campaign."})
assert r6["esito"] == "APPROVATO_MANAGER"
r7 = esegui_finale(GIULIA, "t7", decisione_manager={"approvato": False, "nota": "Budget Q2 esaurito."})
assert r7["esito"] == "RIFIUTATO_MANAGER"

print("\n✅ Tutti i percorsi del grafo funzionano come previsto.")

### Checklist di verifica

Una funzione che controlla i quattro obiettivi leggendo la **struttura** del grafo compilato e i test eseguiti.

In [ ]:
def verifica_obiettivi(app, classe_stato, test_eseguiti: list):
    """Controlla i 4 obiettivi dell'esercitazione. test_eseguiti: [{'nome':..., 'failure': bool, 'ok': bool}]"""
    grafo = app.get_graph()
    nodi = [n for n in grafo.nodes if not n.startswith("__")]
    condizionali = [e for e in grafo.edges if e.conditional]
    controlli = {
        "1. Realizzato con LangGraph (grafo compilato)": hasattr(app, "get_graph") and len(nodi) > 0,
        f"2. Stato strutturato ({len(classe_stato.__annotations__)} campi) e routing condizionale ({len(condizionali)} archi)":
            len(classe_stato.__annotations__) >= 6 and len(condizionali) >= 1,
        f"3. Workflow non banale ({len(nodi)} nodi)": len(nodi) >= 5,
        "4. Failure path testato con successo": any(t["failure"] and t["ok"] for t in test_eseguiti),
    }
    print("=" * 64 + "\n  VERIFICA OBIETTIVI\n" + "=" * 64)
    for descrizione, ok in controlli.items():
        print(f"  {'✅' if ok else '❌'} {descrizione}")
    print(f"\n  📊 {sum(controlli.values())}/4 obiettivi completati")
    return controlli


verifica_obiettivi(app_finale, StatoFinale, [
    {"nome": "badge inesistente",       "failure": True,  "ok": r2["esito"] == "ERRORE"},
    {"nome": "catalogo non disponibile", "failure": True,  "ok": r4["esito"] == "ERRORE"},
    {"nome": "happy path",              "failure": False, "ok": r1["esito"] == "APPROVATO"},
])

### Il tuo caso d'uso

Per portare questo schema su un tuo progetto, segui sempre la stessa traccia:

1. **Descrivi il caso** in poche righe: input, output, esiti possibili, failure path.
2. **Definisci lo stato** (`TypedDict`): campi di input, **un campo per ogni nodo**, campi di controllo (`errore`, `retry_count`), un `log` con `operator.add`.
3. **Scrivi le regole nel codice** (funzioni testabili) e usa il LLM solo per il linguaggio.
4. **Scrivi i nodi** e i **router**; ogni ciclo ha una condizione di uscita.
5. **Aggiungi un `interrupt()`** dove serve un umano (e un checkpointer).
6. **Testa ogni percorso** con `assert`, prima senza LLM, poi con.

Lo scheletro seguente compila già: sostituisci i `TODO`.

In [ ]:
# ── SCHELETRO: il tuo grafo ──────────────────────────────────────────────
class MioStato(TypedDict, total=False):
    input_utente: str
    # TODO: un campo per ogni nodo (es. risultato_nodo_1, risultato_nodo_2)
    errore: str
    retry_count: int
    risposta_finale: str
    log: Annotated[list[str], operator.add]


def mio_nodo_1(stato: MioStato) -> dict:
    # TODO: logica deterministica (o un agente) del primo passo
    return {"log": ["nodo_1 eseguito"]}


def mio_nodo_2(stato: MioStato) -> dict:
    # TODO: secondo passo
    return {"risposta_finale": "TODO: costruisci la risposta", "log": ["nodo_2 eseguito"]}


def mia_gestione_errore(stato: MioStato) -> dict:
    return {"risposta_finale": f"❌ Operazione non completata: {stato.get('errore', 'errore sconosciuto')}"}


def mio_router(stato: MioStato) -> str:
    return "errore" if stato.get("errore") else "avanti"      # TODO: le tue condizioni


mio = StateGraph(MioStato)
mio.add_node("nodo_1", mio_nodo_1)
mio.add_node("nodo_2", mio_nodo_2)
mio.add_node("gestione_errore", mia_gestione_errore)
mio.add_edge(START, "nodo_1")
mio.add_conditional_edges("nodo_1", mio_router, {"avanti": "nodo_2", "errore": "gestione_errore"})
mio.add_edge("nodo_2", END)
mio.add_edge("gestione_errore", END)
mia_app = mio.compile()

print(mia_app.invoke({"input_utente": "prova", "log": []})["log"])

# Parte 6 — Riepilogo e guida di consultazione

## Il percorso in una tabella

| Livello | Strumento | Cosa risolve | Limite che porta al livello successivo |
| :--- | :--- | :--- | :--- |
| 1 | Loop ReAct a mano (`requests`) | capire il tool calling | schemi a mano, tutto da scrivere |
| 2 | **MCP** / **A2A** | standard per collegare tool e agenti | non costruisce l'agente |
| 3 | **LangChain** (`@tool`, `AgentExecutor`) | agente funzionante in poche righe | flusso deciso dal LLM, stato implicito |
| 4 | **LangGraph** | stato esplicito, cicli, persistenza, pause umane | più codice da scrivere |

## Quale pattern scegliere?

| Situazione | Pattern consigliato |
| :--- | :--- |
| Dominio piccolo, prototipo | **Agente singolo** con 2-3 tool (`AgentExecutor`) |
| Competenze separate che cambiano spesso | **Agent-as-Tool** (`Tool.from_function`) |
| Più specialisti da coordinare con un piano | **Supervisor** (LangGraph) |
| Processo con esiti e passaggi definiti | **Grafo esplicito** con router |
| Una decisione costosa o irreversibile | **Human-in-the-Loop** (`interrupt`) |
| Una conversazione che deve ricordare | **Checkpointer** + `thread_id` |
| Un servizio gestito da un altro team/azienda | **A2A** |
| Dati/strumenti riutilizzabili da più applicazioni | **MCP** |

## Cheat-sheet API

| Voglio... | Codice |
| :--- | :--- |
| definire un tool | `@tool` + docstring chiara (type hint obbligatori) |
| testare un tool | `mio_tool.invoke({"param": "valore"})` |
| creare un agente | `create_tool_calling_agent(llm, tools, prompt)` + `AgentExecutor(agent=..., tools=...)` |
| vedere i passi | `return_intermediate_steps=True` |
| osservare i tool | `executor.invoke(input, config={"callbacks": [cb]})` |
| agente come tool | `Tool.from_function(func=lambda q: ex.invoke({"input": q})["output"], name=..., description=...)` |
| grafo con stato | `StateGraph(TypedDict)` + `add_node` + `add_edge` + `compile()` |
| accumulare un campo | `Annotated[list, operator.add]` |
| ramo condizionale | `add_conditional_edges(nodo, router, {chiave: nodo})` |
| parallelismo | due `add_edge` dallo stesso nodo; join con `add_edge([a, b], c)` |
| memoria | `compile(checkpointer=MemorySaver())` + `config={"configurable": {"thread_id": ...}}` |
| ispezionare lo stato | `get_state(config)`, `get_state_history(config)` |
| pausa tra i nodi | `compile(interrupt_before=["nodo"])`, ripresa `stream(None, config)` |
| pausa dentro un nodo | `risposta = interrupt(payload)`, ripresa `invoke(Command(resume=valore), config)` |
| modificare lo stato | `update_state(config, {...})` |

## Errori comuni

| Sintomo | Causa probabile | Rimedio |
| :--- | :--- | :--- |
| `ImportError` su `AgentExecutor` | in LangChain ≥ 1.0 è in `langchain-classic` | `from langchain_classic.agents import ...` |
| Il modello non chiama mai il tool | docstring vaga o prompt senza sequenza esplicita | scrivi **quando** usare il tool e l'ordine dei passi |
| Il modello inventa prezzi o dati | nessuna regola "usa solo i valori dei tool" | aggiungila al prompt e sposta i confronti nel codice |
| `KeyError` / variabile non trovata nel prompt | graffe `{ }` letterali in un `ChatPromptTemplate` | raddoppiale: `{{ }}` |
| Il callback non vede i tool | passato al costruttore di `AgentExecutor` | passalo in `config={"callbacks": [...]}` a `invoke` |
| Loop infinito | nessuna condizione di uscita | `max_iterations` / contatore + `recursion_limit` |
| Errore "node name already used as state key" | nodo e campo dello stato hanno lo stesso nome | rinomina uno dei due |
| `update_state` duplica i messaggi | reducer `operator.add` invece di `add_messages` | usa `add_messages` e lo stesso `id` |
| `interrupt` non funziona | manca il checkpointer o il `thread_id` | `compile(checkpointer=...)` + `config` |
| Effetti ripetuti dopo la ripresa | codice con side effect **prima** di `interrupt()` nello stesso nodo | sposta l'effetto dopo, o rendilo idempotente |
| Risposte diverse a ogni esecuzione | `temperature` alta / modello piccolo | `temperature=0`, regole nel codice, test con `ModelloSimulato` |
| llama3.2 si confonde nei thread lunghi | riferimenti impliciti ("e anche quello?") | ripeti badge e codice prodotto nel messaggio |

## Glossario

| Termine | Significato |
| :--- | :--- |
| **Agente** | LLM + tool + ciclo di ragionamento |
| **Tool calling** | il modello *richiede* una funzione con argomenti strutturati; l'host la esegue |
| **ReAct** | ciclo *Reason ➜ Act ➜ Observe* |
| **MCP** | protocollo agente ➜ tool/dati (Tools, Resources, Prompts) |
| **A2A** | protocollo agente ➜ agente (Agent Card, Task, Artifact) |
| **Agent Card** | scheda JSON che descrive le skill di un agente A2A |
| **Stato** | i dati condivisi da tutti i nodi di un grafo |
| **Reducer** | regola con cui l'output di un nodo si fonde nello stato |
| **Checkpointer** | meccanismo che salva lo stato dopo ogni passo |
| **`thread_id`** | identificatore di una conversazione/esecuzione |
| **Interrupt** | pausa del grafo in attesa di input umano |
| **Fan-out / fan-in** | un nodo che avvia più rami in parallelo / un nodo che aspetta tutti i rami |
| **Failure path** | percorso previsto per gestire un errore invece di andare in crash |
| **Audit trail** | registro delle decisioni (chi, cosa, quando) |

---
*Fine del notebook. Ogni parte è eseguibile in autonomia dopo la Parte 0; le celle 🧠 richiedono Ollama, tutte le altre girano ovunque.*